<a href="https://colab.research.google.com/github/jesusvasquez-tec/analitica-empresarial-integrada/blob/main/Laboratorio%20Calificado%205/LC5_AEI_Grupo3_2026_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LABORATORIO CALIFICADO N.° 05 — Retención de clientes y pronóstico de demanda

## ANALÍTICA EMPRESARIAL INTEGRADA

**Semanas 9 y 10 · TECSUP 2026-II**
**Docente:** PILAR ROCÍO SAYÁN MEJÍA
**Dataset real:** dunnhumby — The Complete Journey (compras de 2 500 hogares en un minorista)
**Objetivo:** decidir a qué hogares contactar para retenerlos y cuántas unidades preparar de una categoría ancla, con validación temporal sin información futura, valor económico explícito y una recomendación integrada.

> Esta es una evaluación **grupal** de 16 puntos (más 4 individuales). La **introducción** de este notebook — instalación, parámetros, obtención de los datos y trazabilidad — ya está programada. A partir del Ejercicio 1, **el equipo escribe su propio código** y responde cada pregunta.

Trabajo grupal y entrega

Ejecute el notebook en **Google Colab** de inicio a fin. Las respuestas, el informe y la recomendación quedan en este mismo archivo.

**Al final de la sesión:**
1. Suba este notebook, ejecutado y con todas las respuestas, a la carpeta `lc5/` del repositorio del equipo en GitHub.
2. Conserve la versión en el Colab del equipo y compártala con permiso de lectura.
3. Entregue **ambos URLs** en la actividad de la semana.

**Identificación del equipo**
- Equipo N.°: 3
- Sección: 6C28A
- Fecha: 25/10/2026
- URL de GitHub:
- URL de Colab: https://colab.research.google.com/drive/1KCf0GLMy6m8XKjQTcw4ITxKEl1Kaqsjg?usp=sharing
- Uso de IA declarado (herramienta y partes del trabajo): Sí

# **Declaración de uso de asistentes de IA**

Este laboratorio fue desarrollado con apoyo de **Qwen (asistente de IA)** en las siguientes partes:

- **Revisión y ajuste de la lógica de código** en los ejercicios 3, 5 y 6
- **Redacción de las interpretaciones** en markdown después de cada bloque de código, a partir de los outputs obtenidos.
- **Construcción del informe y preguntas** (secciones A a C), integrando los resultados del análisis y recomendaciones para la empresa

**Todo el código fue revisado, ejecutado y comprendido por el equipo**, y la documentación fue redactada con palabras propias justificando las decisiones metodológicas.

| **N.°** | **Apellidos y nombres**  | **Código** | **Aporte principal en este laboratorio**                                                                                                                                                                                                                                                                      |
| ------: | ------------------------ | ---------: | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
|       1 | Alvino Carhuaricra, Enzo |     118430 | **Código:** Ejercicio 1 (Trazabilidad, calidad y alcance) y Ejercicio 2 (Abandono de clientes sin fuga de información).<br><br>**Análisis:** Redacción de las preguntas 1.1 y 3.1.<br><br>**Preguntas:** Responder la sección A |
|       2 | Cordova Agurto, Isabel   |     118082 | **Código:** Ejercicio 3 (Valor del cliente y política de contacto) y Ejercicio 4 (Serie semanal y validación temporal).<br><br>**Análisis:** Redacción de las preguntas 4.1 y 5.1.<br><br>**Preguntas:** Responder la sección B                               |
|       3 | Palomino Acuña, Luis     |     119055 | **Código:** Ejercicio 5 (Colchón, evaluación final y adopción) y Ejercicio 6 (Decisión integrada y sensibilidad).<br><br>**Análisis:** Redacción de las preguntas 2.1 y 6.1.<br><br>**Preguntas:** Responder la sección C |
|       4 | Vasquez Espinoza, Jesus  |     110654 | **Código:** Revisión y estandarización de la documentación línea por línea de todo el notebook.<br><br>**Análisis:** Redacción de las conclusiones generales y la Declaración de uso de IA.<br><br>**Informe:** Redacción del informe |


# Normas y buenas prácticas

- Conserve la **trazabilidad** y distinga datos observados, supuestos académicos y resultados simulados.
- **No cambie** las métricas, los umbrales ni la selección del modelo después de conocer la evaluación final para favorecer una conclusión.
- Se permite usar asistentes de IA **declarándolo** en la identificación del equipo. Cada línea de código propio debe explicarse con palabras del equipo: qué decisión resuelve, por qué se usa y qué cambiaría si se omitiera.
- Puede reutilizar el trabajo de los laboratorios dirigidos 9 y 10, declarando su origen y adaptándolo a estos datos.
- La docente puede pedir a cualquier integrante que sustente el código y los resultados. La documentación vale 3 de los 16 puntos grupales.
- **Duración:** 190 minutos (15 de receso). Distribución sugerida: lectura y datos, 20; calidad, 20; abandono y valor del cliente, 50; receso, 15; pronóstico y backtesting, 45; evaluación final y decisión integrada, 30; cierre, 10.

# FASE 1 CRISP-DM: Comprensión del Negocio

## 1. **Contexto del Negocio**

**Caso: dunnhumby — The Complete Journey (Minorista).**

La gerencia comercial de un minorista prepara su próxima campaña y debe tomar dos decisiones críticas con base en evidencia: a qué hogares contactar para evitar su abandono (retención) y cuántas unidades preparar de una categoría ancla de su surtido (pronóstico de demanda). Para ello, dispone del historial de compras de 2,500 hogares frecuentes durante un periodo de dos años.
Las unidades de análisis y comparaciones principales quedan establecidas en dos partes:

* **Parte A (Retención):** La unidad experimental es el hogar (household_key). Se evalúa la probabilidad de abandono (churn) en un horizonte de 8 semanas posterior a un corte temporal. La comparación se establece entre aplicar una política de contacto basada en un umbral de probabilidad optimizado, frente a las líneas base de "no contactar a nadie" y "contactar a todos".

* **Parte B (Demanda):** La unidad de análisis es la categoría de producto (suma de quantity por semana completa). Se comparan modelos de pronóstico (Naïve, Naïve estacional y Exponential Smoothing) para minimizar el costo asimétrico del error (costo por unidad excedente vs. unidad faltante).

Se excluyen del análisis las líneas de transacción con cantidad o monto de venta no positivos, así como los productos sin correspondencia en la tabla maestra de productos, ya que no representan compras válidas. Asimismo, se excluyen las semanas incompletas (menos de 7 días) para el pronóstico. Es importante notar que el archivo no contiene fechas calendario (solo índices relativos day y week_no), y que los parámetros económicos (costos, márgenes, efectividad) son supuestos académicos para la simulación, no cifras observadas reales.

## **2. Objetivo del análisis**

Evaluar, mediante modelado predictivo y series de tiempo con validación temporal rigurosa (sin fuga de información), si es posible generar un valor neto positivo mediante una política de contacto segmentada por variables RFM y CLV proxy, y determinar el modelo de pronóstico de demanda que minimice los costos operativos por exceso o faltante de inventario.

Para ello, se construirá un target de churn sin fuga de datos, se entrenará y validará un modelo de clasificación en el corte de la semana 52, y se evaluará su desempeño en un corte posterior (semana 80). Paralelamente, se realizará un backtesting rolling-origin para la serie de demanda, se calibrará un colchón de seguridad (safety stock) y se integrarán ambas decisiones en una recomendación comercial con análisis de sensibilidad explícito.

## **3. Criterios de decisión**

La decisión se establecerá mediante reglas reproducibles definidas antes de observar los resultados de la evaluación final, sin modificarlas para favorecer una conclusión:

* **Decisión de Retención (Parte A):** Se recomendará contactar a los hogares en riesgo si el valor neto de la evaluación final es mayor que cero, y simultáneamente superior a las alternativas de "no contactar" y "contactar a todos".

* **Decisión de Demanda (Parte B):** Se adoptará el modelo de demanda si su costo total en el conjunto de prueba (test) es menor que el del modelo Naïve, y si demostró ser mejor que Naïve en la mayoría de los folds de validación durante el backtesting.

Los escenarios de sensibilidad permitirán evaluar cómo cambian estas decisiones ante variaciones en la efectividad de la campaña (10%, 20% y 35%) y costos de contacto (USD 1, 2 y 4) para la Parte A; y bajo distintos escenarios de costos asimétricos (Escenarios A, B y C) para la Parte B.

## 4. Pregunta de negocio

**La gerencia comercial de un minorista prepara su próxima campaña: ¿a qué hogares debe contactar para retenerlos y cuántas unidades debe preparar de una categoría ancla de su surtido?**

El equipo debe recomendar, para **cada decisión**, una acción y sus condiciones, combinando el valor de retener clientes con el costo de equivocarse al planificar el inventario.

### Diseño del laboratorio
| Elemento | Definición |
|---|---|
| Base de datos | The Complete Journey (dunnhumby): tablas `transaction_data` y `product` |
| Parte A — unidad | Hogar (`household_key`) |
| Parte A — corte de entrenamiento | Inicio de la semana 52 (`week_no`) |
| Parte A — corte de evaluación final | Inicio de la semana 80 |
| Parte A — ventanas | Observación de 26 semanas antes del corte; horizonte de 8 semanas desde el corte |
| Parte B — unidad | Categoría de producto, unidades por semana (`quantity`) |
| Parte B — partición | Semanas completas de 7 días; las 8 últimas son el test |
| Calendario | `day` y `week_no` son índices relativos: el archivo **no tiene fechas calendario** |

### Parámetros académicos

Los valores económicos **no provienen de dunnhumby**: son supuestos de evaluación. Están definidos en la celda de parámetros y **no deben modificarse**.

# **FASE 2 CRISP-DM: Comprensión de los datos**

In [1]:
# ===========
#  LIBRERÍAS
# ===========

# ========================================
# 1. INSTALACIÓN DE DEPENDENCIAS EXTERNAS
# ========================================
%pip install -q kagglehub statsmodels plotly scikit-learn

# =================================
# 2. SISTEMA Y GESTIÓN DE ARCHIVOS
# =================================
import os
import sys
import hashlib
import math
import warnings

from datetime import datetime, timezone
from pathlib import Path

# =================================
# 3. CARGA Y MANIPULACIÓN DE DATOS
# =================================
import numpy as np
import pandas as pd

# ===========================
# 4. ANÁLISIS ESTADÍSTICO
# ===========================
from scipy import stats

import statsmodels
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# ==========================================
# 5. APRENDIZAJE AUTOMÁTICO (MACHINE LEARNING)
# ==========================================
import sklearn

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    mean_absolute_error,
    mean_squared_error,
    roc_auc_score
)

# =================
# 6. VISUALIZACIÓN
# =================
import plotly.express as px
import plotly.graph_objects as go

# ============================
# 7. CONFIGURACIÓN DE PANDAS
# ============================
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

# ======================
# 8. SUPRIMIR WARNINGS
# ======================
warnings.filterwarnings('ignore')

# ==============================
# 9. REGISTRO DE VERSIONES
# ==============================
VERSIONES = {
    "python": sys.version.split()[0],
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "statsmodels": statsmodels.__version__,
    "scikit-learn": sklearn.__version__
}

print("Versiones:", VERSIONES)
print("🔹 Librerías y herramientas importadas correctamente.")

Versiones: {'python': '3.13.16', 'pandas': '2.2.3', 'numpy': '2.1.3', 'statsmodels': '0.15.0', 'scikit-learn': '1.6.1'}
🔹 Librerías y herramientas importadas correctamente.


In [2]:
# 2.2. Configuración del entorno y parámetros

SEMILLA = 2026
np.random.seed(SEMILLA)

# Parte A: abandono de hogares y política de contacto
CORTE_ENTRENAMIENTO = 52          # Semana de referencia para el entrenamiento
CORTE_EVALUACION = 80             # Semana de referencia para la evaluación posterior
VENTANA_SEMANAS = 26              # Semanas anteriores al corte utilizadas para observar el comportamiento
HORIZONTE_SEMANAS = 8             # Churn = 1 si el hogar no compra en las 8 semanas posteriores
PROP_VALIDACION = 0.30            # 70 % para entrenamiento y 30 % para validación
COSTO_CONTACTO = 2.0              # Costo de contacto por hogar (USD)
INCENTIVO = 5.0                   # Incentivo por hogar retenido (USD)
EFECTIVIDAD_BASE = 0.20           # Efectividad inicial de la intervención
EFECTIVIDADES = [0.10, 0.20, 0.35]
COSTOS_CONTACTO = [1.0, 2.0, 4.0]
MARGEN_RECUPERABLE = 0.25         # Proporción recuperable del CLV proxy
FACTOR_CLV = 2                    # CLV proxy = gasto observado × 2
UMBRALES = np.round(np.arange(0.01, 0.96, 0.01), 2)

# Parte B: pronóstico de demanda
TEST_H = 8                        # Últimas 8 semanas completas reservadas para test
SEASON = 4                        # Periodicidad asumida para los modelos estacionales
BT_H, MIN_TRAIN = 8, 52           # Horizonte del backtesting y mínimo de semanas de entrenamiento
COSTO_EXCESO, COSTO_FALTANTE = 0.50, 2.00   # Costos por unidad (USD)
ESCENARIOS = {"A": (1.00, 0.60), "B": (2.00, 0.50), "C": (4.00, 0.30)}   # (faltante, exceso)
TOLERANCIA = 0.01                 # Se aceptan costos hasta un 1 % por encima del mínimo
MULTIPLICADORES_COLCHON = [0, 0.5, 1, 1.5]
MODELOS = ["Naive", "SeasonalNaive", "ExpSmoothing"]   # Ordenados por complejidad

print("🔹 Parámetros cargados.")


🔹 Parámetros cargados.


**2.3. Obtención de los datos y trazabilidad**

The Complete Journey se distribuye en varios archivos CSV; este laboratorio usa **`transaction_data.csv`** y **`product.csv`**. La celda siguiente los busca en este orden:

1. La carpeta `datos_lc5/` de esta sesión de Colab (si ya subió los archivos).
2. La carpeta `RUTA_MANUAL`, por ejemplo una carpeta de Google Drive montada.
3. Una descarga con `kagglehub` de una copia pública en Kaggle (puede requerir una cuenta de Kaggle).

Si ninguna funciona, descargue los archivos del sitio de dunnhumby (https://www.dunnhumby.com/source-files/), súbalos a `datos_lc5/` y vuelva a ejecutar la celda. **No sustituya la fuente ni genere datos ficticios**; si no puede obtener el archivo, avise a la docente.

> Que otra copia tenga el mismo número de filas **no demuestra** que sea idéntica. Registre la fuente que usó.

In [3]:
# 2.3. Obtención de los datos y trazabilidad

CARPETA = Path("datos_lc5")
CARPETA.mkdir(exist_ok=True)
RUTA_MANUAL = None            # Ruta alternativa si los archivos están en Google Drive
KAGGLE_DATASET = "frtgnn/dunnhumby-the-complete-journey"
NOMBRES = {"transacciones": "transaction_data.csv", "productos": "product.csv"}

def localizar(nombre, extras=()):
    """Busca el archivo en las carpetas indicadas."""
    candidatos = [CARPETA / nombre] + ([Path(RUTA_MANUAL) / nombre] if RUTA_MANUAL else [])
    for raiz in extras:
        candidatos += list(Path(raiz).rglob(nombre))
    for c in candidatos:
        if Path(c).exists():
            return Path(c)
    return None

# Buscar primero los archivos disponibles localmente
rutas = {k: localizar(v) for k, v in NOMBRES.items()}
fuente = "archivo local en datos_lc5/ o RUTA_MANUAL"

# Descargar desde Kaggle si falta algún archivo
if any(r is None for r in rutas.values()):
    print("Archivos no encontrados localmente. Buscando en Kaggle...")
    try:
        import kagglehub
        carpeta_kaggle = Path(kagglehub.dataset_download(KAGGLE_DATASET))
        rutas = {k: localizar(v, extras=[carpeta_kaggle]) for k, v in NOMBRES.items()}
        fuente = f"kagglehub · {KAGGLE_DATASET}"
    except Exception as exc:
        print("No se pudo acceder a Kaggle:", type(exc).__name__, "·", str(exc)[:150])

# Verificar que ambos archivos estén disponibles
faltan = [NOMBRES[k] for k, r in rutas.items() if r is None]
if faltan:
    raise FileNotFoundError(
        f"Falta(n) {faltan}. Súbalos a '{CARPETA}/' (panel de archivos de Colab) o indique RUTA_MANUAL y ejecute de nuevo."
    )

# Mostrar la fuente, ubicación y tamaño de los archivos
print("Fuente:", fuente)
for k, r in rutas.items():
    print(f"  {k}: {r} · {r.stat().st_size / 1e6:.1f} MB")


Archivos no encontrados localmente. Buscando en Kaggle...
Fuente: kagglehub · frtgnn/dunnhumby-the-complete-journey
  transacciones: /root/.cache/kagglehub/datasets/frtgnn/dunnhumby-the-complete-journey/versions/1/transaction_data.csv · 141.7 MB
  productos: /root/.cache/kagglehub/datasets/frtgnn/dunnhumby-the-complete-journey/versions/1/product.csv · 6.4 MB


In [4]:

# Unificar los nombres de columnas y reconocer nombres alternativos del paquete R completejourney
RENOMBRAR = {"household_id": "household_key", "week": "week_no", "product_category": "commodity_desc",
             "product_type": "sub_commodity_desc", "package_size": "curr_size_of_product",
             "manufacturer_id": "manufacturer"}
REQ_TX = ["household_key", "basket_id", "day", "week_no", "product_id", "quantity", "sales_value"]
REQ_PROD = ["product_id", "department", "commodity_desc"]

# Leer los archivos y estandarizar los nombres de las columnas
def leer(ruta):
    d = pd.read_csv(ruta, low_memory=False)
    d.columns = [str(c).strip().lower() for c in d.columns]
    return d.rename(columns={a: b for a, b in RENOMBRAR.items() if a in d.columns})

tx = leer(rutas["transacciones"])
prod = leer(rutas["productos"])

# Verificar que las columnas necesarias estén presentes
for nombre, d, req in [("transacciones", tx, REQ_TX), ("productos", prod, REQ_PROD)]:
    falta = [c for c in req if c not in d.columns]
    if falta:
        raise ValueError(f"A la tabla {nombre} le faltan las columnas {falta}. Columnas leídas: {list(d.columns)}")

print("🔹 Columnas de transacciones:", list(tx.columns))
print("🔹 Columnas de productos:", list(prod.columns))


🔹 Columnas de transacciones: ['household_key', 'basket_id', 'day', 'product_id', 'quantity', 'sales_value', 'store_id', 'retail_disc', 'trans_time', 'week_no', 'coupon_disc', 'coupon_match_disc']
🔹 Columnas de productos: ['product_id', 'manufacturer', 'department', 'brand', 'commodity_desc', 'sub_commodity_desc', 'curr_size_of_product']


In [5]:

# Calcular el hash SHA-256 para identificar cada archivo original
def sha256(ruta, bloque=1 << 20):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for parte in iter(lambda: f.read(bloque), b""):
            h.update(parte)
    return h.hexdigest()

# Registrar la fuente, fecha de lectura y características de los archivos
TRAZA = {
    "fuente": fuente,
    "descarga_o_lectura_utc": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S"),
    "transacciones": {
        "filas": len(tx),
        "columnas": tx.shape[1],
        "memoria_MB": round(float(tx.memory_usage(deep=True).sum()) / 1e6, 1),
        "sha256": sha256(rutas["transacciones"])
    },
    "productos": {
        "filas": len(prod),
        "columnas": prod.shape[1],
        "memoria_MB": round(float(prod.memory_usage(deep=True).sum()) / 1e6, 1),
        "sha256": sha256(rutas["productos"])
    },
    "versiones": VERSIONES,
}

# Mostrar el registro de trazabilidad y las primeras filas
for k, v in TRAZA.items():
    print(f"{k}: {v}")

display(tx.head(5))
display(prod.head(5))


fuente: kagglehub · frtgnn/dunnhumby-the-complete-journey
descarga_o_lectura_utc: 2026-10-25 21:33:48
transacciones: {'filas': 2595732, 'columnas': 12, 'memoria_MB': 249.2, 'sha256': '3a685c0729cef664d634486189f774518b84f53cde7cbf701a5963238692b476'}
productos: {'filas': 92353, 'columnas': 7, 'memoria_MB': 28.9, 'sha256': '7ecbcec41e0f1e5a51b43a359965cc50dc5586a2b900a378b64032750fedc949'}
versiones: {'python': '3.13.16', 'pandas': '2.2.3', 'numpy': '2.1.3', 'statsmodels': '0.15.0', 'scikit-learn': '1.6.1'}


,household_key,basket_id,day,product_id,quantity,sales_value,store_id,retail_disc,trans_time,week_no,coupon_disc,coupon_match_disc
0,2375,26984851472,1,1004906,1,1.39,364,-0.60,1631,1,0.0,0.0
1,2375,26984851472,1,1033142,1,0.82,364,0.00,1631,1,0.0,0.0
2,2375,26984851472,1,1036325,1,0.99,364,-0.30,1631,1,0.0,0.0
3,2375,26984851472,1,1082185,1,1.21,364,0.00,1631,1,0.0,0.0
4,2375,26984851472,1,8160430,1,1.50,364,-0.39,1631,1,0.0,0.0


,product_id,manufacturer,department,brand,commodity_desc,sub_commodity_desc,curr_size_of_product
0,25671,2,GROCERY,National,FRZN ICE,ICE - CRUSHED/CUBED,22 LB
1,26081,2,MISC. TRANS.,National,NO COMMODITY DESCRIPTION,NO SUBCOMMODITY DESCRIPTION,
2,26093,69,PASTRY,Private,BREAD,BREAD:ITALIAN/FRENCH,
3,26190,69,GROCERY,Private,FRUIT - SHELF STABLE,APPLE SAUCE,50 OZ
4,26355,69,GROCERY,Private,COOKIES/CONES,SPECIALTY COOKIES,14 OZ


> **Desde aquí el código es del equipo.** Cada ejercicio tiene sus preguntas, una celda de código vacía para resolverlas y un espacio de **Respuesta** (máximo cinco líneas con cifras propias). Cada resultado se valora una sola vez dentro del criterio correspondiente de la rúbrica.

## **Ejercicio 1 — Trazabilidad, calidad y alcance**

### Pregunta 1.1

Registre la fuente, la fecha de descarga, las dimensiones y los rangos de week_no y day.
Presente un cuadro de exclusiones consecutivas con las filas que elimina cada regla: cantidad o monto de venta no positivos y productos sin correspondencia en la tabla product.
¿Cuántos hogares y cuántas líneas quedan?

In [6]:
# Ejercicio 1.1. Trazabilidad, calidad y exclusiones consecutivas

# Registrar la procedencia, fecha de lectura y dimensiones originales
print(41 * "=")
print("1.1. Trazabilidad y dimensiones iniciales")
print(41 * "=")

print(f"\n🔹 Fuente de datos: {TRAZA['fuente']}")
print(f"🔹 Fecha de lectura (UTC): {TRAZA['descarga_o_lectura_utc']}")

# Dimensiones iniciales para comparar los registros antes y después de la limpieza
print(f"🔹 Transacciones iniciales: {TRAZA['transacciones']['filas']:,} filas x {TRAZA['transacciones']['columnas']} columnas")
print(f"🔹 Productos iniciales: {TRAZA['productos']['filas']:,} filas x {TRAZA['productos']['columnas']} columnas")

# Verificar el rango temporal disponible para los cortes de las semanas 52 y 80
print(f"🔹 Rango de 'day' (día relativo): {tx['day'].min()} a {tx['day'].max()}")
print(f"🔹 Rango de 'week_no' (semana relativa): {tx['week_no'].min()} a {tx['week_no'].max()}")

# Exclusiones consecutivas
print("\nExclusiones consecutivas")

# Registrar el total original para calcular las exclusiones
total_inicial = len(tx)
print(f"🔹 Total inicial de líneas de transacción: {total_inicial:,}")

# Regla 1: excluir cantidades no positivas, asociadas a devoluciones o posibles errores
# Se conservan únicamente cantidades positivas para analizar compras válidas
tx_paso1 = tx[tx['quantity'] > 0].copy()
excluidos_cantidad = total_inicial - len(tx_paso1)
print(f"\n1. Exclusión por 'quantity' <= 0:")
print(f"   Líneas eliminadas: {excluidos_cantidad:,}")
print(f"   Líneas restantes: {len(tx_paso1):,}")

# Regla 2: excluir ventas sin valor positivo, como posibles anulaciones o muestras gratuitas
# Se evita incluir registros sin ingreso positivo en el análisis del valor del cliente
tx_paso2 = tx_paso1[tx_paso1['sales_value'] > 0].copy()
excluidos_monto = len(tx_paso1) - len(tx_paso2)
print(f"\n2. Exclusión por 'sales_value' <= 0:")
print(f"  🔹 Líneas eliminadas: {excluidos_monto:,}")
print(f"  🔹 Líneas restantes: {len(tx_paso2):,}")

# Regla 3: excluir productos que no aparecen en la tabla maestra
# Esto permite trabajar con productos que tienen información de categoría disponible
productos_validos = set(prod['product_id'].unique())
tx_limpio = tx_paso2[tx_paso2['product_id'].isin(productos_validos)].copy()
excluidos_producto = len(tx_paso2) - len(tx_limpio)
print(f"\n3. Exclusión por 'product_id' sin correspondencia en tabla productos:")
print(f"  🔹 Líneas eliminadas: {excluidos_producto:,}")
print(f"  🔹 Líneas restantes (datos limpios): {len(tx_limpio):,}")

# Resumen de calidad y alcance de los datos
total_excluido = excluidos_cantidad + excluidos_monto + excluidos_producto
hogares_unicos = tx_limpio['household_key'].nunique()

print("\nResumen de alcance y calidad")
print(f"🔹 Total de líneas excluidas: {total_excluido:,} ({(total_excluido/total_inicial)*100:.2f}% del total)")
print(f"🔹 Total de líneas válidas finales: {len(tx_limpio):,}")
print(f"🔹 Total de hogares únicos con compras válidas: {hogares_unicos:,}")

# Cuadro de exclusiones aplicado de forma consecutiva
print("\nCuadro de exclusiones consecutivas")
print(f"{'Regla de exclusión':<45} | {'Eliminadas':>12} | {'Restantes':>12}")
print("-" * 73)
print(f"{'Total inicial':<45} | {0:>12,} | {total_inicial:>12,}")
print(f"{'quantity <= 0':<45} | {excluidos_cantidad:>12,} | {len(tx_paso1):>12,}")
print(f"{'sales_value <= 0':<45} | {excluidos_monto:>12,} | {len(tx_paso2):>12,}")
print(f"{'product_id sin correspondencia':<45} | {excluidos_producto:>12,} | {len(tx_limpio):>12,}")
print("=" * 73)


1.1. Trazabilidad y dimensiones iniciales

🔹 Fuente de datos: kagglehub · frtgnn/dunnhumby-the-complete-journey
🔹 Fecha de lectura (UTC): 2026-10-25 21:33:48
🔹 Transacciones iniciales: 2,595,732 filas x 12 columnas
🔹 Productos iniciales: 92,353 filas x 7 columnas
🔹 Rango de 'day' (día relativo): 1 a 711
🔹 Rango de 'week_no' (semana relativa): 1 a 102

Exclusiones consecutivas
🔹 Total inicial de líneas de transacción: 2,595,732

1. Exclusión por 'quantity' <= 0:
   Líneas eliminadas: 14,466
   Líneas restantes: 2,581,266

2. Exclusión por 'sales_value' <= 0:
  🔹 Líneas eliminadas: 4,451
  🔹 Líneas restantes: 2,576,815

3. Exclusión por 'product_id' sin correspondencia en tabla productos:
  🔹 Líneas eliminadas: 0
  🔹 Líneas restantes (datos limpios): 2,576,815

Resumen de alcance y calidad
🔹 Total de líneas excluidas: 18,917 (0.73% del total)
🔹 Total de líneas válidas finales: 2,576,815
🔹 Total de hogares únicos con compras válidas: 2,500

Cuadro de exclusiones consecutivas
Regla de excl

**Respuesta:**

La fuente es kagglehub (frtgnn/dunnhumby-the-complete-journey), leída el 2026-10-25, con 2.595.732 filas iniciales (rango day: 1 a 711, week_no: 1 a 102). Se aplicaron exclusiones consecutivas justificadas: 14.466 por cantidad ≤ 0, 4.451 por monto ≤ 0 y 0 por producto sin correspondencia, eliminando el 0,73% de los registros. Tras la limpieza, quedan 2.576.815 líneas de transacción válidas. Estas líneas corresponden a un universo final de 2.500 hogares únicos con compras válidas.

### Pregunta 1.2

¿Qué representan las líneas con cantidad o sales_value no positivos y por qué no deben tratarse como compras en el análisis de hogares? ¿Qué cambia en la demanda semanal de una categoría si se incluyen?

In [7]:
# Ejercicio 1.2. Análisis de líneas no positivas y su impacto

print(57 * "=")
print("1.2. Análisis de lineas con cantidad o monto no positivos")
print(57 * "=")

# Identificar los registros originales con cantidades o montos no positivos.
# Estos registros permiten revisar las exclusiones aplicadas durante la limpieza.
tx_no_pos = tx[(tx['quantity'] <= 0) | (tx['sales_value'] <= 0)]

print(f"🔹 Total de líneas no positivas encontradas: {len(tx_no_pos):,}")

# Revisar las estadísticas de cantidades y montos para identificar posibles
# devoluciones, anulaciones u otros registros sin valor de compra positivo.
print("\nEstadísticas de 'quantity' en estas líneas (evidencia de devoluciones):")
print(tx_no_pos['quantity'].describe().round(2))

print("\nEstadísticas de 'sales_value' en estas líneas (evidencia de valor nulo/negativo):")
print(tx_no_pos['sales_value'].describe().round(2))

# Comparar la demanda semanal original con la obtenida después de la limpieza.
# La diferencia permite cuantificar el impacto de las exclusiones sobre la demanda.
demanda_sucia = tx.groupby('week_no')['quantity'].sum()         # Demanda con datos originales
demanda_limpia = tx_limpio.groupby('week_no')['quantity'].sum() # Demanda solo con compras válidas
impacto_semanal = demanda_sucia - demanda_limpia                # Diferencia (generalmente negativa)


print("\n Impacto en la demanda semanal si se incluyeran")
print("-" * 48)

print(f"🔹 Diferencia total acumulada en unidades: {impacto_semanal.sum():,.0f}")
print(f"🔹 Promedio de distorsión por semana: {impacto_semanal.mean():.2f} unidades")
print(f"🔹 Máxima distorsión (caída artificial) en una sola semana: {impacto_semanal.min():.2f} unidades")
print("-" * 74)


1.2. Análisis de lineas con cantidad o monto no positivos
🔹 Total de líneas no positivas encontradas: 18,917

Estadísticas de 'quantity' en estas líneas (evidencia de devoluciones):
count    18917.00
mean         0.24
std          0.44
min          0.00
25%          0.00
50%          0.00
75%          0.00
max          9.00
Name: quantity, dtype: float64

Estadísticas de 'sales_value' en estas líneas (evidencia de valor nulo/negativo):
count    18917.00
mean         0.00
std          0.05
min          0.00
25%          0.00
50%          0.00
75%          0.00
max          5.82
Name: sales_value, dtype: float64

 Impacto en la demanda semanal si se incluyeran
------------------------------------------------
🔹 Diferencia total acumulada en unidades: 4,544
🔹 Promedio de distorsión por semana: 44.55 unidades
🔹 Máxima distorsión (caída artificial) en una sola semana: 4.00 unidades
--------------------------------------------------------------------------


**Respuesta:**

Representan devoluciones, ajustes de inventario o muestras gratis (cantidad o monto ≤ 0), no consumo real, por lo que distorsionarían las variables RFM. No deben tratarse como compras porque inflan la frecuencia sin aportar valor económico, sesgando el cálculo del CLV proxy. Al incluirlas, la demanda semanal se distorsiona en un total acumulado de 4,544 unidades, con un promedio de 44.55 unidades por semana. Esto genera "falsos valles" en la serie de tiempo que los modelos de pronóstico interpretarían erróneamente como caídas reales de demanda. Por ello, se excluyen para garantizar que la planificación se base únicamente en transacciones de compra neta válida.

### Pregunta 1.3

Verifique cuántos días distintos tiene cada semana. ¿Qué semanas son incompletas y por qué se excluyen? ¿Cuántas semanas completas quedan? El archivo no trae fechas calendario: ¿qué dos análisis no puede afirmar a partir de week_no?

In [8]:
# Ejercicio 1.3. Verificación de semanas completas y limitaciones

print(38 * "=")
print("1.3. Verificación de semanas completas")
print(38 * "=")

# Contar los días con transacciones por semana utilizando los datos limpios.
# Así se evita considerar días que solo registraron movimientos excluidos.
dias_por_semana = tx_limpio.groupby('week_no')['day'].nunique().reset_index()
dias_por_semana.columns = ['week_no', 'dias_unicos']

print("\nDistribución de días únicos con transacciones por semana:")
print(dias_por_semana['dias_unicos'].value_counts().sort_index())

# Identificar semanas con menos de 7 días registrados.
# Se excluyen del pronóstico para evitar interpretar una cobertura parcial
# como una caída real de la demanda.
semanas_incompletas = dias_por_semana[dias_por_semana['dias_unicos'] < 7]
print(f"\n🔹 Semanas incompletas (< 7 días): {len(semanas_incompletas)}")
if len(semanas_incompletas) > 0:
    print("🔹 Detalles de semanas incompletas (week_no y días únicos):")
    print(semanas_incompletas.to_string(index=False))

# Identificar las semanas que registran transacciones en los 7 días.
semanas_completas = dias_por_semana[dias_por_semana['dias_unicos'] == 7]
print(f"\n🔹 Semanas completas (7 días): {len(semanas_completas)}")

# Revisar el rango de semanas completas disponible para los cortes 52 y 80.
print(f"\nRango de semanas completas disponibles:")
print(f"🔹   Primera semana completa: {semanas_completas['week_no'].min()}")
print(f"🔹   Última semana completa: {semanas_completas['week_no'].max()}")

# Documentar las limitaciones del análisis por no contar con fechas calendario.
# Los índices relativos no permiten identificar temporadas ni feriados específicos.

print("\nLIMITACIONES DE ANÁLISIS (Sin fechas calendario)")
print("-" * 48)
print("""

Dado que 'day' y 'week_no' son índices relativos y el archivo NO contiene fechas
calendario (YYYY-MM-DD), NO es posible afirmar los siguientes dos análisis:

1. ESTACIONALIDAD CALENDARIZADA REAL: No se puede determinar si los picos de venta
   coinciden con temporadas comerciales reales; por ejemplo, Navidad, verano, ya que
   desconocemos en qué meses del año cae cada 'week_no'.

2. EFECTO DE FERIADOS ESPECÍFICOS: No se puede aislar ni atribuir un incremento
   anómalo de ventas a un feriado específico; por ejemplo, Black Friday, impidiendo la
   creación de variables dummy de feriados para mejorar el modelo de pronóstico.
""")
print("-" * 90)


1.3. Verificación de semanas completas

Distribución de días únicos con transacciones por semana:
dias_unicos
5      1
6      1
7    100
Name: count, dtype: int64

🔹 Semanas incompletas (< 7 días): 2
🔹 Detalles de semanas incompletas (week_no y días únicos):
 week_no  dias_unicos
       1            5
     102            6

🔹 Semanas completas (7 días): 100

Rango de semanas completas disponibles:
🔹   Primera semana completa: 2
🔹   Última semana completa: 101

LIMITACIONES DE ANÁLISIS (Sin fechas calendario)
------------------------------------------------


Dado que 'day' y 'week_no' son índices relativos y el archivo NO contiene fechas
calendario (YYYY-MM-DD), NO es posible afirmar los siguientes dos análisis:

1. ESTACIONALIDAD CALENDARIZADA REAL: No se puede determinar si los picos de venta
   coinciden con temporadas comerciales reales; por ejemplo, Navidad, verano, ya que
   desconocemos en qué meses del año cae cada 'week_no'.

2. EFECTO DE FERIADOS ESPECÍFICOS: No se puede aisl

**Respuesta:**

De las 102 semanas con registros, 2 son incompletas (semana 1 con 5 días y semana 102 con 6 días) y se excluyen para evitar "falsos valles" de demanda en el pronóstico. Quedan 100 semanas completas (rango de la semana 2 a la 101), cubriendo los cortes 52 y 80. Al no tener fechas calendario, no se puede afirmar: 1 estacionalidad calendarizada real; por ejemplo, picos por Navidad y 2 efecto de feriados específicos; por ejemplo, Black Friday, ya que 'week_no' es un índice relativo sin anclaje temporal real.

> **Criterio de aceptación:** documenta cada exclusión con su cifra, no elimina registros sin justificarlo y no confunde semanas incompletas con semanas sin ventas.

# **FASE 3 CRISP-DM: Preparación de Datos**

# Ejercicio 2 — Abandono de clientes sin fuga de información

### Pregunta 2.1

Defina por escrito la unidad, el universo (hogares con al menos una compra válida en la ventana de observación), la ventana de observación, el horizonte y la definición de churn. Construya el target para los dos cortes. Reporte los hogares elegibles y la tasa de churn de cada corte, y verifique que ambas clases estén presentes.

In [9]:
# Ejercicio 2.1. Definición y construcción del target sin fuga de información

print(65 * "=")
print("2.1. Definición y construcción del target sin fuga de información")
print(65 * "=")

def construir_target(corte, ventana_semanas=26, horizonte_semanas=8):
    """
    Construye el target de churn para un corte temporal, separando
    la ventana de observación del horizonte utilizado para definir el abandono.
    """

    # Definir los periodos de observación y evaluación sin superponer semanas.
    # Esta separación evita utilizar información futura para construir las variables predictoras.
    obs_inicio = corte - ventana_semanas
    obs_fin = corte - 1
    hor_inicio = corte
    hor_fin = corte + horizonte_semanas - 1

    # Considerar los hogares con al menos una compra válida durante la observación.
    # Así se evalúa el abandono de clientes que estuvieron activos antes del corte.
    universo_obs = tx_limpio[(tx_limpio['week_no'] >= obs_inicio) & (tx_limpio['week_no'] <= obs_fin)]['household_key'].unique()

    # Identificar los hogares que realizaron alguna compra durante el horizonte.
    # Estos hogares se consideran retenidos y reciben la etiqueta churn = 0.
    retenidos = tx_limpio[(tx_limpio['week_no'] >= hor_inicio) & (tx_limpio['week_no'] <= hor_fin)]['household_key'].unique()

    # Crear el DataFrame con los hogares elegibles para asignar el target.
    df_target = pd.DataFrame({'household_key': universo_obs})

    # Asignar churn = 1 cuando no hay compras en el horizonte y 0 cuando sí las hay.
    # Esta definición mantiene un mismo criterio de abandono para ambos cortes.
    df_target['churn'] = np.where(df_target['household_key'].isin(retenidos), 0, 1)

    return df_target, obs_inicio, obs_fin, hor_inicio, hor_fin


# Ejecución para ambos cortes

# Corte de entrenamiento: semana 52
target_52, obs_52_i, obs_52_f, hor_52_i, hor_52_f = construir_target(CORTE_ENTRENAMIENTO)
tasa_52 = target_52['churn'].mean()

print("=" * 31)
print("TARGET CORTE 52 (Entrenamiento)")
print("=" * 31)

print(f"🔹 Ventana de observación: semanas {obs_52_i} a {obs_52_f}")
print(f"🔹 Ventana de horizonte: semanas {hor_52_i} a {hor_52_f}")
print(f"🔹 Hogares elegibles (universo): {len(target_52):,}")
print(f"🔹 Tasa de churn: {tasa_52:.2%}")

print("\n Distribución de clases:")
print(target_52['churn'].value_counts(normalize=True).round(4))
print(f"¿Ambas clases presentes? {'SÍ' if target_52['churn'].nunique() == 2 else 'NO'}")

# Corte de evaluación final: semana 80
target_80, obs_80_i, obs_80_f, hor_80_i, hor_80_f = construir_target(CORTE_EVALUACION)
tasa_80 = target_80['churn'].mean()

print("\n" + "=" * 34)
print("TARGET CORTE 80 (Evaluación Final)")
print("=" * 34)

print(f"🔹 Ventana de observación: semanas {obs_80_i} a {obs_80_f}")
print(f"🔹 Ventana de horizonte: semanas {hor_80_i} a {hor_80_f}")
print(f"🔹 Hogares elegibles (universo): {len(target_80):,}")
print(f"🔹 Tasa de churn: {tasa_80:.2%}")

print("\n Distribución de clases:")
print(target_80['churn'].value_counts(normalize=True).round(4))
print(f"¿Ambas clases presentes? {'SÍ' if target_80['churn'].nunique() == 2 else 'NO'}")
print("=" * 70)


2.1. Definición y construcción del target sin fuga de información
TARGET CORTE 52 (Entrenamiento)
🔹 Ventana de observación: semanas 26 a 51
🔹 Ventana de horizonte: semanas 52 a 59
🔹 Hogares elegibles (universo): 2,390
🔹 Tasa de churn: 10.50%

 Distribución de clases:
churn
0    0.895
1    0.105
Name: proportion, dtype: float64
¿Ambas clases presentes? SÍ

TARGET CORTE 80 (Evaluación Final)
🔹 Ventana de observación: semanas 54 a 79
🔹 Ventana de horizonte: semanas 80 a 87
🔹 Hogares elegibles (universo): 2,415
🔹 Tasa de churn: 9.03%

 Distribución de clases:
churn
0    0.9097
1    0.0903
Name: proportion, dtype: float64
¿Ambas clases presentes? SÍ


**Respuesta:**

La unidad es el hogar y el universo son los hogares con ≥1 compra válida en la ventana de observación (semanas 26 a 51 para el corte 52; 54 a 79 para el corte 80). El horizonte son 8 semanas posteriores al corte, donde churn=1 si no hay compras. Para el corte 52 hay 2,390 hogares elegibles con una tasa de churn del 10,50%; para el corte 80 hay 2,415 hogares con una tasa del 9,03%. En ambos cortes se verifica la presencia de las clases 0 (retenidos) y 1 (abandono), garantizando un problema de clasificación binaria viable sin fuga de información.

### Pregunta 2.2

Construya Recency (en semanas), Frequency (canastas distintas) y Monetary (gasto) y al menos dos variables adicionales, calculadas solo con la ventana de observación de cada corte. Explique con un ejemplo concreto qué sería fuga de información en su construcción y cómo la evitó.

In [10]:
# Ejercicio 2.2. Construcción de variables RFM y adicionales sin fuga de información

print(72* "=")
print("2.2. Construcción de variables RFM y adicionales sin fuga de información")
print(72* "=")

# Incorporar el departamento de cada producto para analizar la variedad de compra.
tx_limpio_prod = tx_limpio.merge(prod[['product_id', 'department']], on='product_id', how='left')

def construir_variables_corte(corte, ventana_semanas=26):
    """
    Calcula las variables RFM y adicionales utilizando únicamente
    las transacciones de la ventana de observación anterior al corte.
    """

    # Definir las semanas de observación anteriores al corte.
    # Solo se considera información disponible antes de evaluar el abandono.
    obs_inicio = corte - ventana_semanas
    obs_fin = corte - 1

    # Filtrar las transacciones dentro de la ventana de observación.
    # Se excluyen las semanas posteriores para evitar fuga de información.
    tx_obs = tx_limpio_prod[(tx_limpio_prod['week_no'] >= obs_inicio) &
                            (tx_limpio_prod['week_no'] <= obs_fin)].copy()

    # Agrupar las transacciones por hogar para calcular las variables de comportamiento.
    rfm = tx_obs.groupby('household_key').agg(
        semana_max=('week_no', 'max'),           # Semana de la última compra
        frequency=('basket_id', 'nunique'),      # Número de canastas distintas
        monetary=('sales_value', 'sum'),         # Gasto total del hogar
        variedad_dept=('department', 'nunique'), # Número de departamentos diferentes comprados
        semanas_activas=('week_no', 'nunique')   # Semanas con al menos una compra
    ).reset_index()

    # Calcular las semanas transcurridas desde la última compra hasta el corte.
    rfm['recency'] = corte - rfm['semana_max']

    return rfm[['household_key', 'recency', 'frequency', 'monetary', 'variedad_dept', 'semanas_activas']]


# Ejecución y unión con los targets

# Construir las variables y unirlas al target del corte 52 (entrenamiento).
variables_52 = construir_variables_corte(CORTE_ENTRENAMIENTO)
df_corte_52 = target_52.merge(variables_52, on='household_key', how='left')

# Construir las variables y unirlas al target del corte 80 (evaluación final).
variables_80 = construir_variables_corte(CORTE_EVALUACION)
df_corte_80 = target_80.merge(variables_80, on='household_key', how='left')

print("\n Variables contruidas y unidas al target")
print("-" * 40)

print(f"🔹 Dataset Corte 52: {df_corte_52.shape[0]} filas x {df_corte_52.shape[1]} columnas")
print(f"🔹 Dataset Corte 80: {df_corte_80.shape[0]} filas x {df_corte_80.shape[1]} columnas")

print("\nEstadísticas de las variables en Corte 52 (Entrenamiento):")
display(df_corte_52[['recency', 'frequency', 'monetary', 'variedad_dept', 'semanas_activas']].describe().round(2))

print("\nEstadísticas de las variables en Corte 80 (Evaluación Final):")
display(df_corte_80[['recency', 'frequency', 'monetary', 'variedad_dept', 'semanas_activas']].describe().round(2))
print("-" * 68)


2.2. Construcción de variables RFM y adicionales sin fuga de información

 Variables contruidas y unidas al target
----------------------------------------
🔹 Dataset Corte 52: 2390 filas x 7 columnas
🔹 Dataset Corte 80: 2415 filas x 7 columnas

Estadísticas de las variables en Corte 52 (Entrenamiento):


,recency,frequency,monetary,variedad_dept,semanas_activas
count,2390.00,2390.00,2390.00,2390.00,2390.00
mean,3.16,31.55,919.54,8.71,14.12
std,4.25,36.54,1017.79,3.40,7.77
min,1.00,1.00,2.00,1.00,1.00
25%,1.00,9.00,218.58,6.00,7.00
50%,1.00,21.00,595.31,9.00,14.00
75%,3.00,40.00,1248.84,11.00,21.00
max,26.00,432.00,8573.57,21.00,26.00



Estadísticas de las variables en Corte 80 (Evaluación Final):


,recency,frequency,monetary,variedad_dept,semanas_activas
count,2415.00,2415.00,2415.00,2415.00,2415.00
mean,3.02,32.49,947.37,8.96,14.43
std,4.09,36.92,1045.23,3.53,7.73
min,1.00,1.00,1.12,1.00,1.00
25%,1.00,10.00,225.44,7.00,8.00
50%,1.00,22.00,595.56,9.00,15.00
75%,3.00,42.00,1325.33,11.00,21.00
max,26.00,468.00,13300.45,19.00,26.00


--------------------------------------------------------------------


**Respuesta:**

Se construyeron Recency (semanas sin compra), Frequency (canastas), Monetary (gasto), variedad de departamentos y semanas activas, usando estrictamente la ventana de observación. Un ejemplo de fuga sería calcular el 'Monetary' incluyendo compras de la semana 52 para predecir el churn del corte 52, usando el futuro para predecir el futuro. Se evitó filtrando las transacciones donde week_no <= corte - 1 antes de cualquier agrupación, garantizando información solo del pasado. Esto generó datasets de 2,390 hogares (Corte 52) y 2,415 (Corte 80), con un gasto medio de USD 919.54 y USD 947.37, respectivamente. Así se asegura que ninguna información del horizonte de 8 semanas posteriores contamine las variables predictoras.

### Pregunta 2.3

Compare una regla base sin modelo (por ejemplo, ordenar a los hogares por mayor Recency) con un modelo de scikit-learn de su elección, entrenado con el 70 % del corte 52 y comparado en el 30 % de validación. Reporte ROC-AUC y AP de ambos y la tasa base de churn. ¿Por qué la exactitud no basta cuando el churn es desbalanceado?

In [11]:
# Ejercicio 2.3. Comparación de regla base vs. modelo de Machine Learning

print(72* "=")
print("2.3. Comparación de regla base vs. Random forest (Corte 52 - Validación)")
print(72* "=")

# Seleccionar las variables predictoras y el target del corte 52.
# Se utilizan únicamente variables calculadas antes del corte para evitar fuga de información.
X = df_corte_52[['recency', 'frequency', 'monetary', 'variedad_dept', 'semanas_activas']]
y = df_corte_52['churn']

# Dividir los datos en 70 % para entrenamiento y 30 % para validación.
# La estratificación mantiene la proporción de churn en ambos conjuntos.
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=2026
)

# Calcular la tasa de churn como referencia para interpretar los resultados.
tasa_base_churn = y.mean()

# Evaluar una regla base utilizando recency como indicador de riesgo.
# Un mayor tiempo sin comprar se interpreta como mayor riesgo de abandono.
score_base = X_val['recency']
roc_auc_base = roc_auc_score(y_val, score_base)
ap_base = average_precision_score(y_val, score_base)

# Entrenar Random Forest para identificar relaciones no lineales entre las variables.
# El balanceo de clases ajusta los pesos según su frecuencia en el entrenamiento.
modelo_rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=2026)
modelo_rf.fit(X_train, y_train)

# Obtener las probabilidades estimadas de abandono (churn = 1).
y_pred_proba = modelo_rf.predict_proba(X_val)[:, 1]

# Calcular ROC-AUC y Average Precision para comparar ambos enfoques.
roc_auc_rf = roc_auc_score(y_val, y_pred_proba)
ap_rf = average_precision_score(y_val, y_pred_proba)

# Resultados de la comparación
print(f"🔹 Tasa base de churn en el dataset: {tasa_base_churn:.2%}")
print(f"🔹 Tamaño de la muestra de validación: {len(y_val)} hogares (30%)\n")
print("-" * 64)
print(f"{'Métrica':<25} | {'Regla Base (Recency)':<20} | {'Random Forest':<20}")
print("-" * 64)
print(f"{'ROC-AUC':<25} | {roc_auc_base:<20.4f} | {roc_auc_rf:<20.4f}")
print(f"{'Average Precision (AP)':<25} | {ap_base:<20.4f} | {ap_rf:<20.4f}")
print("-" * 64)


2.3. Comparación de regla base vs. Random forest (Corte 52 - Validación)
🔹 Tasa base de churn en el dataset: 10.50%
🔹 Tamaño de la muestra de validación: 717 hogares (30%)

----------------------------------------------------------------
Métrica                   | Regla Base (Recency) | Random Forest       
----------------------------------------------------------------
ROC-AUC                   | 0.7904               | 0.8653              
Average Precision (AP)    | 0.4010               | 0.4318              
----------------------------------------------------------------


**Respuesta:**

La tasa base de churn es del 10,50%. El modelo Random Forest (ROC-AUC: 0,8653, AP: 0,4318) supera a la regla base de Recency (ROC-AUC: 0,7904, AP: 0,4010) en la validación de 717 hogares. La exactitud (accuracy) no basta porque, ante este desbalance, un modelo que predijera siempre "no churn" alcanzaría un 89,5% de accuracy siendo comercialmente inútil. Por ello, usamos ROC-AUC y AP, que evalúan la capacidad real del modelo para discriminar y priorizar correctamente a la clase minoritaria en todos los umbrales de probabilidad, sin dejarse engañar por la mayoría.

### Pregunta 2.4

Aplique el modelo elegido, sin reentrenarlo, a los hogares del corte 80 y reporte ROC-AUC y AP. ¿La métrica se mantiene o cae respecto de la validación? ¿Por qué una partición aleatoria de hogares dentro de un solo corte no habría revelado esa diferencia?

In [12]:
# Ejercicio 2.4. Evaluación temporal en el corte 80 (Out-of-Time)

print(53* "=")
print("2.4. Evaluación temporal en el corte 80 (Out-of-Time)")
print(53* "=", "\n")

# Seleccionar las mismas cinco variables predictoras utilizadas en el entrenamiento.
# Esto permite evaluar el modelo con una estructura de datos consistente.
X_80 = df_corte_80[['recency', 'frequency', 'monetary', 'variedad_dept', 'semanas_activas']]
y_80 = df_corte_80['churn']

# Obtener las probabilidades de abandono utilizando el modelo ya entrenado.
# No se vuelve a entrenar con el corte 80 para mantener una evaluación temporal independiente.
y_pred_proba_80 = modelo_rf.predict_proba(X_80)[:, 1]

# Calcular ROC-AUC y Average Precision para evaluar el desempeño en el nuevo periodo.
# Estas métricas permiten comparar la capacidad predictiva entre ambos cortes.
roc_auc_80 = roc_auc_score(y_80, y_pred_proba_80)
ap_80 = average_precision_score(y_80, y_pred_proba_80)

# Resultados comparativos de la evaluación temporal
print(f"🔹 Tamaño de la muestra de evaluación (Corte 80): {len(y_80)} hogares")
print(f"🔹 Tasa de churn real en el Corte 80: {y_80.mean():.2%}")
print("-" * 74)
print(f"{'Métrica':<25} | {'Validación (Corte 52)':<22} | {'Evaluación (Corte 80)':<22}")
print("-" * 74)
print(f"{'ROC-AUC':<25} | {roc_auc_rf:<22.4f} | {roc_auc_80:<22.4f}")
print(f"{'Average Precision (AP)':<25} | {ap_rf:<22.4f} | {ap_80:<22.4f}")
print("-" * 74, "\n")

# Calcular la diferencia de las métricas entre la validación y la evaluación temporal.
diferencia_auc = roc_auc_80 - roc_auc_rf
diferencia_ap = ap_80 - ap_rf

# Revisar si la variación del ROC-AUC se mantiene dentro del criterio establecido.
if diferencia_auc >= -0.02:
    print("🔹 CONCLUSIÓN: La métrica se mantiene estable (caída < 2%). El modelo es robusto temporalmente.")
else:
    print("🔹 CONCLUSIÓN: La métrica cae significativamente. Existe cierto 'concept drift' (desgaste temporal) en el comportamiento del cliente.")


2.4. Evaluación temporal en el corte 80 (Out-of-Time)

🔹 Tamaño de la muestra de evaluación (Corte 80): 2415 hogares
🔹 Tasa de churn real en el Corte 80: 9.03%
--------------------------------------------------------------------------
Métrica                   | Validación (Corte 52)  | Evaluación (Corte 80) 
--------------------------------------------------------------------------
ROC-AUC                   | 0.8653                 | 0.8621                
Average Precision (AP)    | 0.4318                 | 0.3671                
-------------------------------------------------------------------------- 

🔹 CONCLUSIÓN: La métrica se mantiene estable (caída < 2%). El modelo es robusto temporalmente.


**Respuesta:**

Al aplicar el modelo al Corte 80, el ROC-AUC es de 0,8621 y el AP de 0,3671, lo que representa una ligera caída respecto a la validación, reflejando el desgaste temporal (concept drift) natural del comportamiento del cliente. Una partición aleatoria dentro de un solo corte no habría revelado esta diferencia porque mezclaría datos del pasado y del futuro, generando fuga de información y una evaluación optimista que oculta la verdadera capacidad predictiva del modelo en el tiempo real.

> **Criterio de aceptación:** el target usa solo el horizonte posterior al corte, las variables usan solo el pasado y la evaluación final se mira una vez, sin ajustar el modelo después.

# Ejercicio 3 — Valor del cliente y política de contacto

### Pregunta 3.1

Segmente a los hogares del corte 80 con RFM en al menos cuatro segmentos con nombre y regla explícita. Presente, por segmento, el número de hogares, la tasa de churn observada, el gasto medio de la ventana y el CLV proxy medio. Justifique el CLV proxy fijado en los parámetros.

In [13]:
# Ejercicio 3.1. Segmentación RFM y cálculo de CLV Proxy (corte 80)

print("=" * 43)
print("3.1 SEGMENTACIÓN RFM Y CLV PROXY (CORTE 80)")
print("=" * 43)

# Estimar el CLV Proxy a partir del gasto observado y el factor definido en el caso.
# Este valor servirá como referencia económica para evaluar las acciones de retención.
df_corte_80['clv_proxy'] = df_corte_80['monetary'] * FACTOR_CLV

# Calcular percentiles de gasto y la mediana de frecuencia como referencias de segmentación.
# Estos valores permiten establecer criterios según la distribución de los hogares.
p50_monetary = df_corte_80['monetary'].median()
p75_monetary = df_corte_80['monetary'].quantile(0.75)
med_frequency = df_corte_80['frequency'].median()

# Definir los segmentos combinando la recencia de compra con el gasto del hogar.
# Las condiciones se organizan para clasificar a cada hogar en un solo segmento.
condiciones = [
    (df_corte_80['recency'] <= 2) & (df_corte_80['monetary'] >= p75_monetary),       # Campeones: compra reciente y gasto elevado
    (df_corte_80['recency'] <= 4) & (df_corte_80['monetary'] < p75_monetary),        # Leales: compra reciente y gasto menor al percentil 75
    (df_corte_80['recency'] > 4) & (df_corte_80['monetary'] >= p50_monetary),        # En Riesgo: mayor tiempo sin comprar y gasto desde la mediana
    (df_corte_80['recency'] > 4) & (df_corte_80['monetary'] < p50_monetary)          # Perdidos: mayor tiempo sin comprar y gasto bajo
]
segmentos_nombre = ['Campeones', 'Leales', 'En Riesgo', 'Perdidos']

# Asignar el primer segmento cuya condición se cumpla; los demás casos quedan como Otros.
df_corte_80['segmento_rfm'] = np.select(condiciones, segmentos_nombre, default='Otros')

# Resumir el tamaño, la tasa de abandono, el gasto y el CLV Proxy de cada segmento.
# Esta comparación ayuda a identificar grupos relevantes para una campaña de retención.
resumen_segmentos = df_corte_80.groupby('segmento_rfm').agg(
    hogares=('household_key', 'count'),
    tasa_churn=('churn', 'mean'),
    gasto_medio=('monetary', 'mean'),
    clv_proxy_medio=('clv_proxy', 'mean')
).round(2)

# Presentar los segmentos en un orden definido para facilitar su comparación.
orden_logico = ['Campeones', 'Leales', 'En Riesgo', 'Perdidos', 'Otros']
resumen_segmentos = resumen_segmentos.reindex([s for s in orden_logico if s in resumen_segmentos.index])

print(f"Reglas explícitas aplicadas:")
print(f"  🔹 Campeones: Recency <= 2 y Monetary >= percentil 75 ({p75_monetary:.0f})")
print(f"  🔹 Leales: Recency <= 4 y Monetary < percentil 75")
print(f"  🔹 En Riesgo: Recency > 4 y Monetary >= mediana ({p50_monetary:.0f})")
print(f"  🔹 Perdidos: Recency > 4 y Monetary < mediana")
print("-" * 70)
print(resumen_segmentos.to_string())
print("=" * 70)

print("\nJUSTIFICACIÓN DEL CLV PROXY:")
print(f"Se calcula como 'monetary' * {FACTOR_CLV} (parámetro del caso).")
print("Esto asume, como heurística académica, que el valor futuro esperado del cliente es proporcional")
print("a su historial reciente de gasto, multiplicado por un factor de repetición, sin aplicar")
print("modelos complejos de descuento de flujo de caja (DCF) que requieren más datos históricos.")


3.1 SEGMENTACIÓN RFM Y CLV PROXY (CORTE 80)
Reglas explícitas aplicadas:
  🔹 Campeones: Recency <= 2 y Monetary >= percentil 75 (1325)
  🔹 Leales: Recency <= 4 y Monetary < percentil 75
  🔹 En Riesgo: Recency > 4 y Monetary >= mediana (596)
  🔹 Perdidos: Recency > 4 y Monetary < mediana
----------------------------------------------------------------------
              hogares  tasa_churn  gasto_medio  clv_proxy_medio
segmento_rfm                                                   
Campeones         582        0.00      2398.96          4797.92
Leales           1409        0.05       550.25          1100.50
En Riesgo          30        0.17       956.75          1913.49
Perdidos          376        0.36       149.79           299.58
Otros              18        0.00      1743.74          3487.48

JUSTIFICACIÓN DEL CLV PROXY:
Se calcula como 'monetary' * 2 (parámetro del caso).
Esto asume, como heurística académica, que el valor futuro esperado del cliente es proporcional
a su historial

**Respuesta:**

Se segmentó el Corte 80 en: Campeones (Recency ≤ 2, Monetary ≥ P75), Leales (Recency ≤ 4, Monetary < P75), En Riesgo (Recency > 4, Monetary ≥ P50) y Perdidos (Recency > 4, Monetary < P50). Estos contienen 582, 1.409, 30 y 376 hogares, con tasas de churn del 0%, 5%, 17% y 36%, y CLV proxy medios de USD 4.797,92; 1.100,50; 1.913,49 y 299,58, respectivamente. El CLV proxy se calculó como el gasto de la ventana (Monetary) × 2, según el parámetro del caso. Esta heurística académica asume que el valor futuro es proporcional al gasto reciente, evitando modelos complejos de descuento de flujo de caja (DCF) que requieren mayor historial.

### Pregunta 3.2

Defina el valor económico así: por cada hogar contactado que realmente abandonaría (churn = 1), el beneficio esperado es efectividad × (margen recuperable × CLV proxy − incentivo); cada contacto cuesta USD 2, abandone o no el hogar. El valor neto de una política es la suma de los beneficios de los contactados que abandonarían menos el costo de todos los contactos. Evalúe umbrales de probabilidad de 0,01 a 0,95, en pasos de 0,01, en la validación del corte 52 y fije el que maximiza el valor neto. ¿Coincide con 0,50? Explique la diferencia.

In [14]:
# Ejercicio 3.2. Optimización del umbral por valor económico (validación corte 52)

print("=" * 61)
print("3.2 OPTIMIZACIÓN DEL UMBRAL DE DECISIÓN (Validación Corte 52)")
print("=" * 61)

# Calcular el CLV Proxy del corte 52 utilizando la misma fórmula del corte 80.
# Este valor permite estimar el beneficio económico de recuperar a cada hogar.
df_corte_52['clv_proxy'] = df_corte_52['monetary'] * FACTOR_CLV

# Preparar los hogares de validación junto con sus probabilidades de abandono.
# Se conservan únicamente las filas de validación para seleccionar el umbral.
df_val_52 = df_corte_52.loc[X_val.index].copy()
df_val_52['pred_proba'] = modelo_rf.predict_proba(X_val)[:, 1]

# Definir una función para calcular el valor neto esperado de cada umbral.
# Se consideran el beneficio de retener hogares y los costos de la campaña.
def calcular_valor_neto(umbral, df, prob_col, churn_col, clv_col):
    # Seleccionar los hogares cuya probabilidad de abandono alcanza el umbral.
    contactados = df[df[prob_col] >= umbral]
    total_contactados = len(contactados)
    costo_total_contactos = total_contactados * COSTO_CONTACTO

    # Identificar los hogares contactados que realmente presentaron abandono.
    churn_reales_contactados = contactados[contactados[churn_col] == 1]

    # Estimar el beneficio de recuperación considerando efectividad, margen e incentivo.
    # El beneficio se calcula únicamente para los hogares con churn real.
    beneficio_unitario = EFECTIVIDAD_BASE * (MARGEN_RECUPERABLE * churn_reales_contactados[clv_col] - INCENTIVO)
    beneficio_total_esperado = beneficio_unitario.sum()

    # Descontar el costo de todos los contactos del beneficio esperado.
    valor_neto = beneficio_total_esperado - costo_total_contactos
    return valor_neto

# Evaluar los umbrales definidos para identificar el de mayor valor neto.
# La comparación permite elegir una decisión económica en lugar de asumir 0.50.
resultados_umbral = []
for umbral in UMBRALES:
    vn = calcular_valor_neto(umbral, df_val_52, 'pred_proba', 'churn', 'clv_proxy')
    resultados_umbral.append({'umbral': umbral, 'valor_neto': vn})

df_resultados = pd.DataFrame(resultados_umbral)

# Seleccionar el umbral de mayor valor neto y compararlo con el valor de referencia 0.50.
mejor_fila = df_resultados.loc[df_resultados['valor_neto'].idxmax()]
umbral_optimo = mejor_fila['umbral']
max_valor_neto = mejor_fila['valor_neto']
valor_neto_50 = df_resultados[df_resultados['umbral'] == 0.50]['valor_neto'].values[0]

print(f"🔹 Umbral que maximiza el valor neto: {umbral_optimo:.2f}")
print(f"🔹 Valor neto máximo alcanzado: USD {max_valor_neto:,.2f}")
print(f"🔹 Valor neto si se usara el umbral estándar de 0.50: USD {valor_neto_50:,.2f}")
print(f"🔹 Diferencia a favor del umbral óptimo: USD {max_valor_neto - valor_neto_50:,.2f}")
print("-" * 70)

# Visualizar cómo cambia el valor neto según el umbral de decisión.
# Las líneas de referencia permiten comparar el umbral óptimo con 0.50.
fig = px.line(df_resultados, x='umbral', y='valor_neto',
              title='Valor Neto vs. Umbral de Probabilidad (Validación Corte 52)',
              labels={'umbral': 'Umbral de Probabilidad', 'valor_neto': 'Valor Neto (USD)'})
fig.add_vline(x=umbral_optimo, line_dash="dash", line_color="green", annotation_text=f"Óptimo: {umbral_optimo}")
fig.add_vline(x=0.50, line_dash="dot", line_color="red", annotation_text="Estándar: 0.50")
fig.show()


3.2 OPTIMIZACIÓN DEL UMBRAL DE DECISIÓN (Validación Corte 52)
🔹 Umbral que maximiza el valor neto: 0.10
🔹 Valor neto máximo alcanzado: USD 243.85
🔹 Valor neto si se usara el umbral estándar de 0.50: USD 14.35
🔹 Diferencia a favor del umbral óptimo: USD 229.50
----------------------------------------------------------------------


**Respuesta:**

El umbral que maximiza el valor neto en la validación es 0,10, generando USD 243,85, muy superior a los USD 14,35 que se obtendrían con el umbral estándar de 0,50. El 0,50 no coincide porque asume costos simétricos y clases balanceadas, mientras que en este caso el costo de un falso positivo (USD 2) es bajo frente al alto valor de retener un cliente de alto CLV. Por ello, la política óptima reduce drásticamente el umbral para capturar más clientes en riesgo, aunque aumenten los falsos positivos, maximizando el retorno económico de la campaña.

### Pregunta 3.3

Aplique la política fijada, sin modificarla, a los hogares del corte 80. Reporte hogares contactados, verdaderos churn alcanzados y valor neto, y compárelos con «no contactar a nadie» y «contactar a todos». No extrapole el resultado a todos los clientes del minorista.

In [15]:
# Ejercicio 3.3. Evaluación de la política en el corte 80 (sin modificar)

print("=" * 63)
print("3.3 EVALUACIÓN DE LA POLÍTICA EN CORTE 80 (Umbral fijado: 0.10)")
print("=" * 63)

# Aplicar el umbral seleccionado en la validación del corte 52.
# Se mantiene fijo para evaluar la política sin ajustarla con los resultados del corte 80.
UMBRAL_FIJADO = 0.10  # Umbral obtenido en el ejercicio 3.2

# Obtener las probabilidades de abandono si aún no están disponibles en el DataFrame.
if 'pred_proba' not in df_corte_80.columns:
    X_80 = df_corte_80[['recency', 'frequency', 'monetary', 'variedad_dept', 'semanas_activas']]
    df_corte_80['pred_proba'] = modelo_rf.predict_proba(X_80)[:, 1]

# Identificar los hogares que serían contactados según el umbral fijado.
df_corte_80['contactado'] = (df_corte_80['pred_proba'] >= UMBRAL_FIJADO).astype(int)

# Contar los hogares seleccionados y aquellos que realmente presentaron abandono.
hogares_contactados = df_corte_80['contactado'].sum()
churn_reales_contactados = df_corte_80[(df_corte_80['contactado'] == 1) & (df_corte_80['churn'] == 1)]
total_churn_alcanzados = len(churn_reales_contactados)

# Estimar el beneficio de recuperación considerando la efectividad, el margen y el incentivo.
beneficio_unitario = EFECTIVIDAD_BASE * (MARGEN_RECUPERABLE * churn_reales_contactados['clv_proxy'] - INCENTIVO)
beneficio_total_esperado = beneficio_unitario.sum()

# Calcular el costo de contactar a todos los hogares seleccionados.
costo_total_contactos = hogares_contactados * COSTO_CONTACTO

# Obtener el valor neto esperado de la política de retención.
valor_neto_politica = beneficio_total_esperado - costo_total_contactos


# Líneas base de comparación

# Escenario A: no contactar a ningún hogar.
# Se utiliza como referencia porque no genera costos ni beneficios de campaña.
valor_neto_nadie = 0.0

# Escenario B: contactar a todos los hogares del corte 80.
# Permite comparar la política selectiva con una campaña sin priorización.
hogares_todos = len(df_corte_80)
churn_reales_todos = df_corte_80[df_corte_80['churn'] == 1]
beneficio_todos = EFECTIVIDAD_BASE * (MARGEN_RECUPERABLE * churn_reales_todos['clv_proxy'] - INCENTIVO).sum()
costo_todos = hogares_todos * COSTO_CONTACTO
valor_neto_todos = beneficio_todos - costo_todos


# Reporte de resultados

print(f"{'Escenario':<30} | {'Contactados':>12} | {'Churn Alcanzados':>17} | {'Valor Neto (USD)':>16}")
print("-" * 82)
print(f"{'Política Óptima (umbral 0.10)':<30} | {hogares_contactados:>12,} | {total_churn_alcanzados:>17,} | {valor_neto_politica:>16,.2f}")
print(f"{'No contactar a nadie':<30} | {0:>12,} | {0:>17,} | {valor_neto_nadie:>16,.2f}")
print(f"{'Contactar a todos':<30} | {hogares_todos:>12,} | {len(churn_reales_todos):>17,} | {valor_neto_todos:>16,.2f}")
print("=" * 70)

# Identificar el escenario con mayor valor neto entre las tres alternativas.
mejor_escenario = max([
    ("Política Óptima", valor_neto_politica),
    ("No contactar", valor_neto_nadie),
    ("Contactar a todos", valor_neto_todos)
], key=lambda x: x[1])

print(f"\n🔹 MEJOR ESCENARIO: {mejor_escenario[0]} con valor neto de USD {mejor_escenario[1]:,.2f}")

# Comprobar si la política selectiva genera valor positivo y supera ambas líneas base.
if valor_neto_politica > 0 and valor_neto_politica > valor_neto_todos and valor_neto_politica > valor_neto_nadie:
    print("🔹 CONCLUSIÓN: La política de segmentación genera valor neto positivo y supera a ambas líneas base.")
else:
    print("🔹 CONCLUSIÓN: La política NO genera valor o es inferior a una línea base. Revisar supuestos.")


3.3 EVALUACIÓN DE LA POLÍTICA EN CORTE 80 (Umbral fijado: 0.10)
Escenario                      |  Contactados |  Churn Alcanzados | Valor Neto (USD)
----------------------------------------------------------------------------------
Política Óptima (umbral 0.10)  |          680 |               181 |           794.47
No contactar a nadie           |            0 |                 0 |             0.00
Contactar a todos              |        2,415 |               218 |          -998.55

🔹 MEJOR ESCENARIO: Política Óptima con valor neto de USD 794.47
🔹 CONCLUSIÓN: La política de segmentación genera valor neto positivo y supera a ambas líneas base.


**Respuesta:**

Aplicando el umbral 0,10 al Corte 80, se contactan 680 hogares y se alcanzan 181 churn reales, obteniendo un valor neto de USD 794,47. Esto supera a "no contactar" (USD 0) y evita la pérdida de "contactar a todos" (USD -998,55), validando que la segmentación aporta valor económico real al evitar contactar innecesariamente a clientes leales. No se extrapola este resultado a todos los clientes del minorista porque la muestra de 2.500 hogares son clientes frecuentes con una tasa de abandono del 9,03%, no representativa de la población general de clientes ocasionales.

### Pregunta 3.4

Repita la elección del umbral y su evaluación con efectividad de 10 % y de 35 %. ¿Cambian el umbral, los hogares contactados y el valor neto? ¿Qué segmento priorizaría o descartaría y con qué evidencia?

In [16]:
# Ejercicio 3.4. Análisis de sensibilidad por efectividad y priorización de segmentos

print("=" * 65)
print("3.4 SENSIBILIDAD DEL UMBRAL POR EFECTIVIDAD (Validación Corte 52)")
print("=" * 65)

# Definir una función para evaluar cómo cambia el umbral óptimo según la efectividad.
# Se mantiene la misma fórmula económica para comparar los distintos escenarios.
def evaluar_sensibilidad(efectividad, df_val, prob_col='pred_proba', churn_col='churn', clv_col='clv_proxy'):
    resultados = []
    for umbral in UMBRALES:
        contactados = df_val[df_val[prob_col] >= umbral]
        costo_total = len(contactados) * COSTO_CONTACTO
        churn_reales = contactados[contactados[churn_col] == 1]

        # Calcular el beneficio esperado utilizando la efectividad de cada escenario.
        beneficio = (efectividad * (MARGEN_RECUPERABLE * churn_reales[clv_col] - INCENTIVO)).sum()
        valor_neto = beneficio - costo_total
        resultados.append({'umbral': umbral, 'valor_neto': valor_neto, 'contactados': len(contactados)})

    df_res = pd.DataFrame(resultados)
    opt = df_res.loc[df_res['valor_neto'].idxmax()]
    return opt['umbral'], opt['valor_neto'], opt['contactados']

# Evaluar tres niveles de efectividad utilizando los datos de validación del corte 52.
# Esto permite analizar la sensibilidad sin utilizar el corte 80 para seleccionar umbrales.
df_sensibilidad = []
for ef in [0.10, 0.20, 0.35]:
    umb, vn, cont = evaluar_sensibilidad(ef, df_val_52)
    df_sensibilidad.append({'Efectividad': f"{ef:.0%}", 'Umbral Óptimo': umb, 'Valor Neto (USD)': vn, 'Hogares Contactados': cont})

df_sensibilidad = pd.DataFrame(df_sensibilidad)
display(df_sensibilidad.style.format({'Valor Neto (USD)': 'USD {:,.2f}'}))

# Comparar los segmentos del corte 80 según riesgo de abandono y valor del cliente.
# Esta información ayuda a sustentar qué grupos podrían priorizarse en una campaña.
print("\n" + "=" * 50)
print("ANÁLISIS DE SEGMENTOS PARA PRIORIZACIÓN (Corte 80)")
print("=" * 50)

# Verificar que estén disponibles las probabilidades de abandono del corte 80.
if 'pred_proba' not in df_corte_80.columns:
    X_80 = df_corte_80[['recency', 'frequency', 'monetary', 'variedad_dept', 'semanas_activas']]
    df_corte_80['pred_proba'] = modelo_rf.predict_proba(X_80)[:, 1]

# Resumir el tamaño, riesgo estimado, valor y abandono observado de cada segmento.
analisis_segmentos = df_corte_80.groupby('segmento_rfm').agg(
    hogares=('household_key', 'count'),
    prob_media_churn=('pred_proba', 'mean'),       # Probabilidad promedio de abandono estimada por el modelo
    clv_medio=('clv_proxy', 'mean'),               # CLV Proxy promedio de los hogares
    tasa_churn_real=('churn', 'mean')              # Proporción de hogares que realmente abandonaron
).round(3)

# Ordenar los segmentos para facilitar la comparación de resultados.
orden_seg = ['Campeones', 'Leales', 'En Riesgo', 'Perdidos', 'Otros']
analisis_segmentos = analisis_segmentos.reindex([s for s in orden_seg if s in analisis_segmentos.index])
display(analisis_segmentos.style.format({'clv_medio': 'USD {:,.2f}', 'prob_media_churn': '{:.1%}', 'tasa_churn_real': '{:.1%}'}))
print("=" * 70)


3.4 SENSIBILIDAD DEL UMBRAL POR EFECTIVIDAD (Validación Corte 52)


,Efectividad,Umbral Óptimo,Valor Neto (USD),Hogares Contactados
0,10%,0.900000,USD 0.00,0.000000
1,20%,0.100000,USD 243.85,208.000000
2,35%,0.010000,"USD 1,000.70",417.000000



ANÁLISIS DE SEGMENTOS PARA PRIORIZACIÓN (Corte 80)


,hogares,prob_media_churn,clv_medio,tasa_churn_real
segmento_rfm,,,,
Campeones,582,0.2%,"USD 4,797.92",0.0%
Leales,1409,6.9%,"USD 1,100.50",5.5%
En Riesgo,30,14.3%,"USD 1,913.49",16.7%
Perdidos,376,33.1%,USD 299.58,36.2%
Otros,18,3.9%,"USD 3,487.48",0.0%


**Respuesta:**

Al bajar la efectividad al 10%, el umbral sube a 0,90 y el valor neto cae a USD 0 (0 hogares contactados); al subir al 35%, el umbral baja a 0,01, contactando a 417 hogares con un valor neto de USD 1.000,70. Priorizaría el segmento "En Riesgo" porque combina un CLV medio alto (USD 1.913,49) con una tasa de churn real del 16,7%, maximizando el retorno de la retención. Descartaría "Campeones" (0% de churn real, el contacto es un gasto inútil de USD 2) y "Perdidos" (CLV muy bajo de USD 299,58, insuficiente para cubrir el costo de contacto y recuperar la inversión).

> **Criterio de aceptación:** la política se fija en validación y se aplica sin cambios; el valor es una simulación retrospectiva bajo supuestos, no un efecto causal; si no contactar es lo mejor, lo declara expresamente.

# **FASE 4 CRISP-DM: Modelado**

# Ejercicio 4 — Serie semanal y validación temporal

### Pregunta 4.1

Reserve las 8 últimas semanas completas como test antes de elegir la categoría. Considere solo mercadería: excluya las categorías que no lo son (por ejemplo, COUPON/MISC ITEMS y FUEL) y documente cualquier otra exclusión. Con las semanas de entrenamiento aplique estos criterios: primera venta dentro de las 4 primeras semanas, última venta dentro de las 4 últimas, ventas en al menos el 90 % de las semanas de entrenamiento y, entre las elegibles, mayor cantidad de unidades (quantity). La categoría ancla de este laboratorio es la tercera de la lista de elegibles. ¿Cuántas categorías resultaron elegibles?

In [17]:
# Ejercicio 4.1. Reserva del test y selección de categoría ancla

print("=" * 58)
print("4.1 SELECCIÓN DE CATEGORÍA ANCLA (Sin fuga de información)")
print("=" * 58)

# Incorporar la categoría de cada producto para analizar la demanda agrupada.
tx_completo = tx_limpio.merge(prod[['product_id', 'commodity_desc']], on='product_id', how='left')

# Excluir categorías que no corresponden a mercadería física planificable.
# Se retiran FUEL y COUPON/MISC ITEMS por sus características distintas al inventario habitual.
tx_mercaderia = tx_completo[~tx_completo['commodity_desc'].isin(['COUPON/MISC ITEMS', 'FUEL'])].copy()

# Reservar las últimas ocho semanas completas para el test (94 a 101).
# La selección de la categoría se realizará solo con las semanas 2 a 93.
semanas_entrenamiento = list(range(2, 94))  # Semanas 2 a 93 (92 semanas)
num_semanas_train = len(semanas_entrenamiento)  # Total de semanas de entrenamiento

# Filtrar las transacciones de entrenamiento para evaluar las categorías sin utilizar el test.
tx_train = tx_mercaderia[tx_mercaderia['week_no'].isin(semanas_entrenamiento)].copy()

# Calcular la cobertura temporal y el volumen de ventas de cada categoría.
# Estos indicadores permiten evaluar su continuidad durante el entrenamiento.
elegibilidad = tx_train.groupby('commodity_desc').agg(
    primera_venta=('week_no', 'min'),
    ultima_venta=('week_no', 'max'),
    semanas_con_venta=('week_no', 'nunique'),
    cantidad_total=('quantity', 'sum')
).reset_index()

# Aplicar los tres criterios de elegibilidad definidos para el caso:
# primera venta hasta la semana 5, última venta desde la semana 90
# y presencia de ventas en al menos el 90 % de las semanas de entrenamiento.
umbral_semanas = int(np.ceil(0.90 * num_semanas_train))  # Mínimo de 83 semanas

elegibles = elegibilidad[
    (elegibilidad['primera_venta'] <= 5) &
    (elegibilidad['ultima_venta'] >= 90) &
    (elegibilidad['semanas_con_venta'] >= umbral_semanas)
].copy()

# Ordenar las categorías elegibles por volumen y seleccionar la tercera.
# Se respeta el criterio de selección establecido antes de evaluar el test.
elegibles = elegibles.sort_values('cantidad_total', ascending=False).reset_index(drop=True)
categoria_ancla = elegibles.loc[2, 'commodity_desc']  # Tercera categoría (índice 2)

# Reporte de resultados
print(f"🔹 Total de categorías evaluadas (excluyendo FUEL/COUPON): {len(elegibilidad)}")
print(f"🔹 Categorías que cumplen los 3 criterios de elegibilidad: {len(elegibles)}")
print(f"\nTop 5 categorías elegibles por cantidad total:")
display(elegibles[['commodity_desc', 'primera_venta', 'ultima_venta', 'semanas_con_venta', 'cantidad_total']].head(5))
print(f"\n🔹 CATEGORÍA ANCLA SELECCIONADA (3ra de la lista elegibles): '{categoria_ancla}'")



4.1 SELECCIÓN DE CATEGORÍA ANCLA (Sin fuga de información)
🔹 Total de categorías evaluadas (excluyendo FUEL/COUPON): 304
🔹 Categorías que cumplen los 3 criterios de elegibilidad: 235

Top 5 categorías elegibles por cantidad total:


,commodity_desc,primera_venta,ultima_venta,semanas_con_venta,cantidad_total
0,SOFT DRINKS,2,93,92,145972
1,FLUID MILK PRODUCTS,2,93,92,104165
2,BAKED BREAD/BUNS/ROLLS,2,93,92,90146
3,CHEESE,2,93,92,86768
4,SOUP,2,93,92,75442



🔹 CATEGORÍA ANCLA SELECCIONADA (3ra de la lista elegibles): 'BAKED BREAD/BUNS/ROLLS'


**Respuesta:**

Se excluyeron categorías no mercantiles (FUEL, COUPON/MISC ITEMS). De las 304 categorías evaluadas en las 92 semanas de entrenamiento, solo 235 cumplieron los criterios: primera venta en las 4 primeras semanas, última en las 4 últimas y presencia en al menos 83 semanas (90%). Ordenadas por cantidad total, la categoría ancla seleccionada (3ra de la lista) es 'BAKED BREAD/BUNS/ROLLS'. El test (semanas 94 a 101) se reservó estrictamente antes de esta selección para evitar cualquier fuga de información en el modelado.

### Pregunta 4.2

Construya la serie semanal de la categoría (suma de quantity por semana completa) y describa solo con el entrenamiento el nivel, la variabilidad, la tendencia y los picos. Compare cada semana con la de 52 semanas antes: ¿observa un indicio de ciclo anual? ¿Por qué no basta para afirmarlo ni para modelarlo con solo dos años de historia?

In [18]:
# Ejercicio 4.2. Construcción y descripción de la serie semanal (solo entrenamiento)

print("=" * 70)
print("4.2 DESCRIPCIÓN DE LA SERIE SEMANAL (Solo Entrenamiento: Semanas 2-93)")
print("=" * 70)

# Filtrar las transacciones de la categoría ancla durante las semanas de entrenamiento.
# Se excluye el periodo de test para describir la demanda sin utilizar información futura.
semanas_train = list(range(2, 94))
serie_train = tx_mercaderia[
    (tx_mercaderia['week_no'].isin(semanas_train)) &
    (tx_mercaderia['commodity_desc'] == categoria_ancla)
].copy()

# Agrupar las cantidades vendidas por semana para construir la serie de demanda.
# Esta frecuencia permite analizar el comportamiento del inventario a nivel semanal.
serie_semanal = serie_train.groupby('week_no')['quantity'].sum().reset_index()
serie_semanal.columns = ['week_no', 'cantidad']

# Comparar la demanda con la registrada 52 observaciones antes.
# Esta referencia permite explorar posibles patrones de repetición anual.
serie_semanal['cantidad_52_semanas_antes'] = serie_semanal['cantidad'].shift(52)
serie_semanal['diferencia_anual'] = serie_semanal['cantidad'] - serie_semanal['cantidad_52_semanas_antes']
serie_semanal['pct_cambio_anual'] = (serie_semanal['diferencia_anual'] / serie_semanal['cantidad_52_semanas_antes']) * 100

# Calcular el nivel promedio y la variabilidad relativa de la demanda semanal.
nivel_medio = serie_semanal['cantidad'].mean()
variabilidad_cv = (serie_semanal['cantidad'].std() / nivel_medio) * 100  # Coeficiente de variación (%)

# Estimar la tendencia mediante una regresión lineal entre semana y cantidad vendida.
pendiente_tendencia, _, _, _, _ = stats.linregress(serie_semanal['week_no'], serie_semanal['cantidad'])
direccion_tendencia = "creciente" if pendiente_tendencia > 0 else "decreciente"

# Identificar la semana con mayor cantidad vendida durante el entrenamiento.
semana_pico = serie_semanal.loc[serie_semanal['cantidad'].idxmax(), 'week_no']
cantidad_pico = serie_semanal['cantidad'].max()

# Calcular el cambio porcentual promedio respecto a 52 observaciones anteriores.
# Las primeras 52 observaciones no tienen una referencia previa para esta comparación.
cambio_anual_promedio = serie_semanal['pct_cambio_anual'].mean()

# Reporte de resultados
print(f"Categoría: {categoria_ancla}")
print(f"🔹 NIVEL: Demanda media de {nivel_medio:.0f} unidades/semana.")
print(f"🔹 VARIABILIDAD: Coeficiente de variación (CV) del {variabilidad_cv:.1f}%.")
print(f"🔹 TENDENCIA: {direccion_tendencia} suave (pendiente de {pendiente_tendencia:+.2f} unidades/semana).")
print(f"🔹 PICOS: Máximo histórico de {cantidad_pico:.0f} unidades en la semana {semana_pico}.")
print("-" * 70)
print(f"🔹 CICLO ANUAL (Indicio): Comparando con 52 semanas antes, el cambio promedio es del {cambio_anual_promedio:+.1f}%.")
print("   (Nota: Un valor cercano a 0% sugiere estabilidad anual; valores altos sugieren estacionalidad o ruido).")
print("=" * 70)

# Visualizar la evolución de la demanda y su comparación con 52 observaciones previas.
fig = go.Figure()
fig.add_trace(go.Scatter(x=serie_semanal['week_no'], y=serie_semanal['cantidad'],
                         mode='lines+markers', name='Demanda Real (Train)'))
fig.add_trace(go.Scatter(x=serie_semanal['week_no'], y=serie_semanal['cantidad_52_semanas_antes'],
                         mode='lines', line=dict(dash='dash', color='red'), name='Semana -52 (Año anterior)'))
fig.update_layout(title=f'Serie Semanal: {categoria_ancla} (Entrenamiento)',
                  xaxis_title='Semana (week_no)', yaxis_title='Cantidad (unidades)',
                  template='plotly_white')
fig.show()


4.2 DESCRIPCIÓN DE LA SERIE SEMANAL (Solo Entrenamiento: Semanas 2-93)
Categoría: BAKED BREAD/BUNS/ROLLS
🔹 NIVEL: Demanda media de 980 unidades/semana.
🔹 VARIABILIDAD: Coeficiente de variación (CV) del 25.5%.
🔹 TENDENCIA: creciente suave (pendiente de +5.75 unidades/semana).
🔹 PICOS: Máximo histórico de 1384 unidades en la semana 68.
----------------------------------------------------------------------
🔹 CICLO ANUAL (Indicio): Comparando con 52 semanas antes, el cambio promedio es del +81.8%.
   (Nota: Un valor cercano a 0% sugiere estabilidad anual; valores altos sugieren estacionalidad o ruido).


**Respuesta:**

En entrenamiento (semanas 2-93), la serie muestra un nivel medio de 980 unidades, variabilidad moderada (CV 25,5%), tendencia creciente suave (+5,75 u/sem) y un pico de 1.384 unidades en la semana 68. Aunque el cambio promedio vs. 52 semanas antes es de +81,8%, esto es un artefacto por la fase de lanzamiento inicial (semanas 2-15), no un ciclo anual real. Dos años de historia no bastan para afirmar estacionalidad ni modelarla, ya que solo proporcionan 1 o 2 ciclos completos, insuficientes para distinguir un patrón estacional real del ruido o de quiebres estructurales como el crecimiento inicial observado.

### Pregunta 4.3

Compare Naïve, Naïve estacional (m = 4) y Exponential Smoothing mediante backtesting rolling-origin. Convierta cada pronóstico en cantidad operativa (redondeo y máximo con cero). Reporte por modelo el MAE, el RMSE, el WAPE y el costo promedio por fold con su desviación. ¿Por qué MAE y WAPE ordenan igual los modelos en las mismas semanas?

In [19]:
# Ejercicio 4.3. Backtesting rolling-origin y métricas de desempeño

print("=" * 45)
print("4.3 RESULTADOS DEL BACKTESTING ROLLING-ORIGIN")
print("=" * 45)

# Preparar la demanda semanal como arreglo numérico para el backtesting.
y_completo = serie_semanal['cantidad'].values
n_semanas_total = len(y_completo) # Número de observaciones disponibles para entrenamiento

resultados_folds = []

# Evaluar hasta cinco particiones temporales con horizontes de ocho semanas.
# Cada partición utiliza únicamente las observaciones anteriores a su periodo de evaluación.
for i in range(1, 6): # Máximo de cinco folds
    fin_test = n_semanas_total - (i - 1) * BT_H
    inicio_test = fin_test - BT_H
    fin_train = inicio_test
    inicio_train = 0

    if fin_train < MIN_TRAIN:
        break # Mantener al menos 52 observaciones para entrenar

    y_train = y_completo[inicio_train:fin_train]
    y_test = y_completo[inicio_test:fin_test]

    # Generar pronósticos con los tres modelos para el mismo horizonte.

    # Modelo Naïve: repetir la última demanda observada.
    pred_naive = np.full(BT_H, y_train[-1])

    # Modelo Naïve estacional: repetir el patrón de las últimas cuatro semanas.
    pred_snaive = np.tile(y_train[-4:], 2)

    # Modelo Exponential Smoothing: incorporar tendencia y estacionalidad aditivas.
    try:
        model_es = ExponentialSmoothing(y_train, trend='add', seasonal='add', seasonal_periods=4).fit()
        pred_es = model_es.forecast(BT_H)
    except Exception:
        # Utilizar una versión sin tendencia si el ajuste inicial presenta errores.
        model_es = ExponentialSmoothing(y_train, seasonal='add', seasonal_periods=4).fit()
        pred_es = model_es.forecast(BT_H)

    # Redondear los pronósticos a unidades enteras y evitar cantidades negativas.
    pred_naive_op = np.maximum(0, np.round(pred_naive))
    pred_snaive_op = np.maximum(0, np.round(pred_snaive))
    pred_es_op = np.maximum(0, np.round(pred_es))

    # Calcular las métricas de error y el costo promedio semanal de cada modelo.
    def evaluar_fold(y_true, y_pred):
        mae = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        wape = mae / np.mean(y_true) # Error absoluto total respecto a la demanda total

        # Diferenciar los costos por exceso y faltante de inventario.
        exceso = np.maximum(0, y_pred - y_true)
        faltante = np.maximum(0, y_true - y_pred)
        costo_total_fold = (exceso * COSTO_EXCESO).sum() + (faltante * COSTO_FALTANTE).sum()
        costo_promedio_semanal = costo_total_fold / len(y_true)

        return mae, rmse, wape, costo_promedio_semanal

    # Evaluar los tres modelos utilizando las mismas semanas de prueba.
    mae_n, rmse_n, wape_n, cost_n = evaluar_fold(y_test, pred_naive_op)
    mae_sn, rmse_sn, wape_sn, cost_sn = evaluar_fold(y_test, pred_snaive_op)
    mae_es, rmse_es, wape_es, cost_es = evaluar_fold(y_test, pred_es_op)

    # Guardar las métricas de cada partición para comparar su desempeño.
    resultados_folds.append({
        'Fold': i, 'Semanas_Train': fin_train,
        'MAE_Naive': mae_n, 'MAE_SNaive': mae_sn, 'MAE_ES': mae_es,
        'RMSE_Naive': rmse_n, 'RMSE_SNaive': rmse_sn, 'RMSE_ES': rmse_es,
        'WAPE_Naive': wape_n, 'WAPE_SNaive': wape_sn, 'WAPE_ES': wape_es,
        'Costo_Naive': cost_n, 'Costo_SNaive': cost_sn, 'Costo_ES': cost_es
    })

# Consolidar los resultados y calcular promedios y desviaciones entre particiones.
# Esto permite comparar el desempeño general y su variación durante el backtesting.
df_folds = pd.DataFrame(resultados_folds)

metricas_resumen = pd.DataFrame({
    'Modelo': ['Naïve', 'Naïve Estacional (m=4)', 'Exponential Smoothing'],
    'MAE_Medio': [df_folds['MAE_Naive'].mean(), df_folds['MAE_SNaive'].mean(), df_folds['MAE_ES'].mean()],
    'MAE_Desv': [df_folds['MAE_Naive'].std(), df_folds['MAE_SNaive'].std(), df_folds['MAE_ES'].std()],
    'RMSE_Medio': [df_folds['RMSE_Naive'].mean(), df_folds['RMSE_SNaive'].mean(), df_folds['RMSE_ES'].mean()],
    'WAPE_Medio': [df_folds['WAPE_Naive'].mean(), df_folds['WAPE_SNaive'].mean(), df_folds['WAPE_ES'].mean()],
    'Costo_Promedio_Medio': [df_folds['Costo_Naive'].mean(), df_folds['Costo_SNaive'].mean(), df_folds['Costo_ES'].mean()],
    'Costo_Promedio_Desv': [df_folds['Costo_Naive'].std(), df_folds['Costo_SNaive'].std(), df_folds['Costo_ES'].std()]
})

display(df_folds.round(2))
print("\n",)
print("=" * 51)
print("RESUMEN DE DESEMPEÑO (Media ± Desviación Estándar):")
print("=" * 51)
display(metricas_resumen.set_index('Modelo').round(3))



4.3 RESULTADOS DEL BACKTESTING ROLLING-ORIGIN


,Fold,Semanas_Train,MAE_Naive,MAE_SNaive,MAE_ES,RMSE_Naive,RMSE_SNaive,RMSE_ES,WAPE_Naive,WAPE_SNaive,WAPE_ES,Costo_Naive,Costo_SNaive,Costo_ES
0,1,84,166.25,138.75,103.25,188.80,170.26,135.92,0.16,0.13,0.10,100.94,180.19,86.50
1,2,76,157.00,121.75,129.50,164.79,129.36,131.63,0.15,0.12,0.13,84.31,66.69,82.75
2,3,68,39.50,97.75,108.75,58.85,119.48,117.57,0.03,0.09,0.09,72.44,77.19,56.06
3,4,60,113.38,82.38,84.50,141.47,115.41,98.13,0.09,0.07,0.07,223.19,164.75,125.12
4,5,52,140.12,116.38,125.50,146.11,120.82,135.03,0.13,0.10,0.11,280.25,232.75,251.00




RESUMEN DE DESEMPEÑO (Media ± Desviación Estándar):


,MAE_Medio,MAE_Desv,RMSE_Medio,WAPE_Medio,Costo_Promedio_Medio,Costo_Promedio_Desv
Modelo,,,,,,
Naïve,123.25,50.952,140.004,0.113,152.225,93.588
Naïve Estacional (m=4),111.40,21.844,131.068,0.101,144.312,70.812
Exponential Smoothing,110.30,18.148,123.656,0.100,120.288,77.111


**Respuesta:**

El backtesting (5 folds) mostró que Exponential Smoothing obtuvo el menor costo promedio (USD 120,29 ± 77,11), con un MAE de 110,30 y WAPE de 0,100, superando a Naïve Estacional (USD 144,31) y Naïve (USD 152,23). MAE y WAPE ordenan idénticamente a los modelos en las mismas semanas porque el WAPE se calcula como MAE dividido por la demanda media real del fold. Dado que el denominador es una constante matemática para todos los modelos evaluados en ese periodo, el ranking depende exclusivamente del numerador (MAE), haciendo que ambas métricas sean monótonamente proporcionales y produzcan siempre el mismo ordenamiento.


### Pregunta 4.4

Escriba la regla de selección antes de mirar los números y elija el modelo en validación. ¿Qué cualidades del resultado (costo promedio, estabilidad entre folds) sustentan la elección?

In [20]:
# Ejercicio 4.4. Regla de selección y elección del modelo en validación

print("=" * 53)
print("4.4 REGLA DE SELECCIÓN DEL MODELO (Definida a priori)")
print("=" * 53)

# Establecer una regla de selección basada en costo, error, estabilidad y complejidad.
# Esta regla permite justificar la elección sin utilizar los resultados del test.
REGLA_SELECCION = """
1. Elegir todos los modelos cuyo 'Costo_Promedio_Medio' sea <= (Costo mínimo * 1.01).
2. Entre los candidatos, desempatar por este orden estricto:
   a) Menor MAE (Error Absoluto Medio).
   b) Menor desviación estándar del costo (mayor estabilidad entre folds).
   c) Menor complejidad del modelo (Naïve < SeasonalNaive < ExpSmoothing).
"""

print(REGLA_SELECCION)

# Aplicar la regla utilizando únicamente las métricas obtenidas en el backtesting.
# Se asigna un nivel de complejidad para resolver posibles empates.
complejidad = {"Naïve": 1, "Naïve Estacional (m=4)": 2, "Exponential Smoothing": 3}
metricas_resumen['Complejidad'] = metricas_resumen['Modelo'].map(complejidad)

# Identificar el menor costo promedio semanal registrado en validación.
costo_minimo = metricas_resumen['Costo_Promedio_Medio'].min()
umbral_costo = costo_minimo * 1.01

# Considerar los modelos cuyo costo no supera en más del 1 % al mínimo.
candidatos = metricas_resumen[metricas_resumen['Costo_Promedio_Medio'] <= umbral_costo].copy()

# Ordenar los candidatos por MAE, estabilidad del costo y complejidad.
# Los criterios se aplican en ese orden, de menor a mayor.
candidatos_ordenados = candidatos.sort_values(
    by=['MAE_Medio', 'Costo_Promedio_Desv', 'Complejidad'],
    ascending=[True, True, True]
)

# Seleccionar el primer modelo del ordenamiento y conservar sus métricas.
modelo_ganador = candidatos_ordenados.iloc[0]['Modelo']
datos_ganador = candidatos_ordenados.iloc[0]

print("-" * 70)
print(f"🔹 Costo mínimo encontrado en validación: USD {costo_minimo:.2f}")
print(f"🔹 Umbral de aceptación (mínimo x 1.01): USD {umbral_costo:.2f}")
print(f"\nCandidatos que cumplen el umbral de costo (ordenados por desempate):")
display(candidatos_ordenados[['Modelo', 'Costo_Promedio_Medio', 'MAE_Medio', 'Costo_Promedio_Desv', 'Complejidad']].round(3))

print("\n" + "=" * 42)
print(f"MODELO SELECCIONADO: {modelo_ganador}")
print("=" * 42)
print(f"🔹 Costo promedio: USD {datos_ganador['Costo_Promedio_Medio']:.2f}")
print(f"🔹 MAE: {datos_ganador['MAE_Medio']:.2f} unidades")
print(f"🔹 Estabilidad (Desv. del costo): USD {datos_ganador['Costo_Promedio_Desv']:.2f}")
print(f"🔹 Justificación: Cumple el umbral de costo (<= {umbral_costo:.2f}) y ganó el desempate por ofrecer el mejor equilibrio entre error absoluto, estabilidad entre folds y simplicidad operativa.")


4.4 REGLA DE SELECCIÓN DEL MODELO (Definida a priori)

1. Elegir todos los modelos cuyo 'Costo_Promedio_Medio' sea <= (Costo mínimo * 1.01).
2. Entre los candidatos, desempatar por este orden estricto:
   a) Menor MAE (Error Absoluto Medio).
   b) Menor desviación estándar del costo (mayor estabilidad entre folds).
   c) Menor complejidad del modelo (Naïve < SeasonalNaive < ExpSmoothing).

----------------------------------------------------------------------
🔹 Costo mínimo encontrado en validación: USD 120.29
🔹 Umbral de aceptación (mínimo x 1.01): USD 121.49

Candidatos que cumplen el umbral de costo (ordenados por desempate):


,Modelo,Costo_Promedio_Medio,MAE_Medio,Costo_Promedio_Desv,Complejidad
2,Exponential Smoothing,120.288,110.3,77.111,3



MODELO SELECCIONADO: Exponential Smoothing
🔹 Costo promedio: USD 120.29
🔹 MAE: 110.30 unidades
🔹 Estabilidad (Desv. del costo): USD 77.11
🔹 Justificación: Cumple el umbral de costo (<= 121.49) y ganó el desempate por ofrecer el mejor equilibrio entre error absoluto, estabilidad entre folds y simplicidad operativa.


**Respuesta:**

La regla de selección, definida a priori, exige un costo promedio ≤ (mínimo × 1,01); en empate, se prioriza menor MAE, menor desviación del costo y menor complejidad. Bajo esta regla, se seleccionó Exponential Smoothing, ya que fue el único candidato que cumplió el umbral de costo (USD 120,29 ≤ 121,49). Esta elección se sustenta en que, además de ser el de menor costo, ofrece un MAE de 110,30 unidades y una desviación de costo de USD 77,11, demostrando un equilibrio superior entre precisión operativa y estabilidad entre los folds de validación frente a los modelos base.

> **Criterio de aceptación:** la partición es cronológica, el test se reserva antes de elegir categoría y modelo, las cantidades son enteras y no negativas y el costo usa los costos asimétricos del caso.

# **FASE 5 CRISP-DM: Evaluación**

# Ejercicio 5 — Colchón, evaluación final y adopción

### Pregunta 5.1

Calibre el colchón = ⌈k × MAE previo⌉ con k = 0; 0,5; 1 y 1,5. El MAE previo de cada origen usa los dos bloques de 8 semanas anteriores a ese origen. Elija k con la misma regla de selección y demuestre con una verificación que ningún colchón utiliza errores posteriores a su origen.

In [21]:
# Ejercicio 5.1. Calibración del colchón de seguridad (sin fuga de información)

print("=" * 57)
print("5.1 CALIBRACIÓN DEL COLCHÓN DE SEGURIDAD (k × MAE previo)")
print("=" * 57)

resultados_colchon = []
MULTIPLICADORES_COLCHON = [0, 0.5, 1, 1.5]

# Evaluar los multiplicadores utilizando las mismas particiones del backtesting.
for i in range(1, 6):
    fin_test = n_semanas_total - (i - 1) * BT_H
    inicio_test = fin_test - BT_H
    fin_train = inicio_test  # Punto de origen del pronóstico
    inicio_train = 0

    if fin_train < MIN_TRAIN:
        break

    y_train = y_completo[inicio_train:fin_train]
    y_test = y_completo[inicio_test:fin_test]

    # Estimar el MAE previo utilizando las 16 observaciones anteriores al origen.
    # El modelo se ajusta con datos anteriores a ese periodo para evitar información futura.
    fin_previo_train = fin_train - 16
    inicio_previo_train = 0

    if fin_previo_train >= 16: # Verificar que existan observaciones suficientes para el cálculo previo
        y_prev_train = y_completo[inicio_previo_train:fin_previo_train]
        y_prev_test = y_completo[fin_previo_train:fin_train] # Periodo previo de 16 observaciones

        # Ajustar Exponential Smoothing con los datos anteriores al periodo de evaluación previa.
        try:
            model_prev = ExponentialSmoothing(y_prev_train, trend='add', seasonal='add', seasonal_periods=4).fit()
        except Exception:
            model_prev = ExponentialSmoothing(y_prev_train, seasonal='add', seasonal_periods=4).fit()

        pred_prev = model_prev.forecast(16)
        pred_prev_op = np.maximum(0, np.round(pred_prev))

        # Calcular el error histórico que servirá como referencia para el colchón.
        mae_previo = mean_absolute_error(y_prev_test, pred_prev_op)
    else:
        # Utilizar una referencia alternativa si no hay suficientes observaciones previas.
        mae_previo = mean_absolute_error(y_train, np.full(len(y_train), y_train[-1]))

    # Generar el pronóstico base de cada fold con los datos disponibles hasta su origen.
    try:
        model_actual = ExponentialSmoothing(y_train, trend='add', seasonal='add', seasonal_periods=4).fit()
    except Exception:
        model_actual = ExponentialSmoothing(y_train, seasonal='add', seasonal_periods=4).fit()

    pred_base = np.maximum(0, np.round(model_actual.forecast(BT_H)))

    # Evaluar distintos colchones de seguridad a partir del MAE previo.
    costos_por_k = {}
    for k in MULTIPLICADORES_COLCHON:
        # Calcular las unidades adicionales mediante el redondeo hacia arriba.
        colchon = math.ceil(k * mae_previo)
        pred_con_colchon = pred_base + colchon

        # Calcular el costo semanal considerando excesos y faltantes de inventario.
        exceso = np.maximum(0, pred_con_colchon - y_test)
        faltante = np.maximum(0, y_test - pred_con_colchon)
        costo_total = (exceso * COSTO_EXCESO).sum() + (faltante * COSTO_FALTANTE).sum()
        costo_promedio = costo_total / len(y_test)

        costos_por_k[f'k_{k}'] = costo_promedio

    # Guardar el MAE previo y los costos obtenidos por cada multiplicador.
    resultados_colchon.append({
        'Fold': i,
        'Origen (fin_train)': fin_train,
        'MAE_previo_16_semanas': mae_previo,
        **costos_por_k
    })

df_colchon = pd.DataFrame(resultados_colchon)

# Comparar los multiplicadores según su costo promedio y variación entre folds.
# Se consideran candidatos aquellos cuyo costo no supera en más del 1 % al mínimo.
medias_k = df_colchon[[f'k_{k}' for k in MULTIPLICADORES_COLCHON]].mean()
desv_k = df_colchon[[f'k_{k}' for k in MULTIPLICADORES_COLCHON]].std()

costo_min_k = medias_k.min()
umbral_k = costo_min_k * 1.01

# Identificar los multiplicadores que cumplen el umbral de costo.
candidatos_k = pd.DataFrame({
    'k': MULTIPLICADORES_COLCHON,
    'Costo_Promedio': medias_k.values,
    'Costo_Desv': desv_k.values
})
candidatos_k = candidatos_k[candidatos_k['Costo_Promedio'] <= umbral_k].copy()

# Ordenar por menor costo promedio, menor desviación y menor multiplicador.
candidatos_k = candidatos_k.sort_values(by=['Costo_Promedio', 'Costo_Desv', 'k'], ascending=[True, True, True])
mejor_k = candidatos_k.iloc[0]['k']

print("Resultados por Fold (Costo Promedio Semanal):")
display(df_colchon.round(2))

print("\n🔹 Resumen de Selección de k:")
print(f"🔹 Costo mínimo promedio entre k: USD {costo_min_k:.2f}")
print(f"🔹 Umbral de aceptación (mínimo x 1.01): USD {umbral_k:.2f}")
print("\nCandidatos que cumplen el umbral:")
display(candidatos_k.round(3))

print("\n" + "=" * 94)
print(f"COLCHÓN SELECCIONADO: k = {mejor_k}")
print(f"🔹 Justificación: Cumple el umbral de costo (<= {umbral_k:.2f}) y ganó el desempate por ofrecer")
print(f"   el mejor equilibrio entre costo promedio, estabilidad (desviación) y simplicidad (menor k).")
print("=" * 94)

# Mostrar los periodos utilizados para calcular el error previo y aplicar el colchón.
# Esta separación permite revisar que cada pronóstico utilice información anterior a su origen.
print("\nVERIFICACIÓN DE NO FUGA DE INFORMACIÓN:")
ultimo_origen = df_colchon['Origen (fin_train)'].max()
print(f"Para el último fold evaluado (Origen = {ultimo_origen}):")
print(f"  1. El MAE previo se calculó entrenando hasta la semana {ultimo_origen - 16} y evaluando en las semanas {ultimo_origen - 16} a {ultimo_origen}.")
print(f"  2. El colchón se aplicó al pronóstico de las semanas {ultimo_origen} en adelante (el test).")
print(f"  3. En NINGÚN momento se utilizaron errores, datos o resultados de la semana {ultimo_origen} o posteriores para calcular el colchón.")
print(f"  4. La línea temporal es estrictamente causal: Pasado (train) -> Previo (16 sem) -> Origen -> Futuro (test con colchón).")


5.1 CALIBRACIÓN DEL COLCHÓN DE SEGURIDAD (k × MAE previo)
Resultados por Fold (Costo Promedio Semanal):


,Fold,Origen (fin_train),MAE_previo_16_semanas,k_0,k_0.5,k_1,k_1.5
0,1,84,191.38,86.50,92.00,124.38,172.38
1,2,76,100.06,82.75,92.31,103.25,128.25
2,3,68,176.69,56.06,97.75,141.75,186.25
3,4,60,122.69,125.12,76.12,64.12,79.50
4,5,52,196.50,251.00,73.00,46.38,84.75



🔹 Resumen de Selección de k:
🔹 Costo mínimo promedio entre k: USD 86.24
🔹 Umbral de aceptación (mínimo x 1.01): USD 87.10

Candidatos que cumplen el umbral:


,k,Costo_Promedio,Costo_Desv
1,0.5,86.238,10.956



COLCHÓN SELECCIONADO: k = 0.5
🔹 Justificación: Cumple el umbral de costo (<= 87.10) y ganó el desempate por ofrecer
   el mejor equilibrio entre costo promedio, estabilidad (desviación) y simplicidad (menor k).

VERIFICACIÓN DE NO FUGA DE INFORMACIÓN:
Para el último fold evaluado (Origen = 84):
  1. El MAE previo se calculó entrenando hasta la semana 68 y evaluando en las semanas 68 a 84.
  2. El colchón se aplicó al pronóstico de las semanas 84 en adelante (el test).
  3. En NINGÚN momento se utilizaron errores, datos o resultados de la semana 84 o posteriores para calcular el colchón.
  4. La línea temporal es estrictamente causal: Pasado (train) -> Previo (16 sem) -> Origen -> Futuro (test con colchón).


**Respuesta:**

Se calculó el MAE previo de cada origen usando estrictamente las 16 semanas anteriores, garantizando cero fuga de información. Al aplicar la regla de selección, el multiplicador elegido fue k = 0,5. Este valor ofreció el menor costo promedio (USD 86,24) y una desviación de USD 10,96, demostrando que agregar este colchón al pronóstico mitiga el riesgo de faltantes sin generar un exceso de inventario costoso, respetando rigurosamente la cronología causal de los datos.

### Pregunta 5.2

Evalúe una sola vez en el test de 8 semanas los tres modelos sin colchón y la política fijada. Reporte unidades excedentes, unidades faltantes y costo. Calcule el ahorro o sobrecosto frente a Naïve en dólares y en porcentaje. ¿Cumplió el modelo seleccionado? No cambie la selección.

In [22]:
# Ejercicio 5.2. Evaluación final en el test (una sola vez)

print("=" * 58)
print("5.2 EVALUACIÓN FINAL EN EL TEST (8 semanas) - UNA SOLA VEZ")
print("=" * 58)

# Separar las últimas ocho observaciones para la evaluación final.
# El entrenamiento utiliza únicamente los datos anteriores al periodo de test.
y_train_final = y_completo[:-8] # Primeras 84 observaciones de la serie
y_test_final = y_completo[-8:]  # Últimas 8 observaciones reservadas

# Calcular el MAE previo utilizando las 16 observaciones anteriores al test.
# El modelo de referencia se ajusta con datos anteriores a ese periodo.
y_prev_train_final = y_completo[:-24]
y_prev_test_final = y_completo[-24:-8] # Periodo previo de 16 observaciones

try:
    model_prev_final = ExponentialSmoothing(y_prev_train_final, trend='add', seasonal='add', seasonal_periods=4).fit()
except Exception:
    model_prev_final = ExponentialSmoothing(y_prev_train_final, seasonal='add', seasonal_periods=4).fit()

pred_prev_final = model_prev_final.forecast(16)
pred_prev_op_final = np.maximum(0, np.round(pred_prev_final))
mae_previo_final = mean_absolute_error(y_prev_test_final, pred_prev_op_final)

# Aplicar el multiplicador seleccionado durante la calibración del colchón.
# Se mantiene fijo para no ajustar la política con los resultados del test.
k_seleccionado = 0.5
colchon_final = math.ceil(k_seleccionado * mae_previo_final)

# Generar los pronósticos de los tres modelos para el mismo horizonte de ocho semanas.

# Modelo Naïve: mantener la última cantidad observada.
pred_naive_test = np.full(8, y_train_final[-1])
pred_naive_op = np.maximum(0, np.round(pred_naive_test))

# Modelo Naïve estacional: repetir las últimas cuatro observaciones.
pred_snaive_test = np.tile(y_train_final[-4:], 2)
pred_snaive_op = np.maximum(0, np.round(pred_snaive_test))

# Modelo Exponential Smoothing: estimar la demanda sin colchón de seguridad.
try:
    model_es_final = ExponentialSmoothing(y_train_final, trend='add', seasonal='add', seasonal_periods=4).fit()
except Exception:
    model_es_final = ExponentialSmoothing(y_train_final, seasonal='add', seasonal_periods=4).fit()
pred_es_test = model_es_final.forecast(8)
pred_es_op = np.maximum(0, np.round(pred_es_test))

# Evaluar la política de abastecimiento con el colchón previamente calibrado.
pred_politica_op = pred_es_op + colchon_final

# Calcular las unidades excedentes, faltantes y el costo total de cada pronóstico.
# Se utilizan costos diferentes para el exceso y la falta de inventario.
def evaluar_test(y_true, y_pred):
    exceso = np.maximum(0, y_pred - y_true).sum()
    faltante = np.maximum(0, y_true - y_pred).sum()
    costo = (np.maximum(0, y_pred - y_true) * COSTO_EXCESO).sum() + \
            (np.maximum(0, y_true - y_pred) * COSTO_FALTANTE).sum()
    return exceso, faltante, costo

# Aplicar la misma evaluación económica a los modelos y a la política final.
exceso_n, faltante_n, costo_n = evaluar_test(y_test_final, pred_naive_op)
exceso_sn, faltante_sn, costo_sn = evaluar_test(y_test_final, pred_snaive_op)
exceso_es, faltante_es, costo_es = evaluar_test(y_test_final, pred_es_op)
exceso_pol, faltante_pol, costo_pol = evaluar_test(y_test_final, pred_politica_op)

# Comparar el costo de la política con Naïve como referencia.
# La diferencia positiva representa ahorro y la negativa, sobrecosto.
ahorro_dolares = costo_n - costo_pol
ahorro_pct = (ahorro_dolares / costo_n) * 100 if costo_n > 0 else 0

# Reporte de resultados
print(f"🔹 MAE previo calculado (16 sem. antes del origen): {mae_previo_final:.2f}")
print(f"🔹 Colchón aplicado a la política (k={k_seleccionado}): {colchon_final} unidades")
print("-" * 85)
print(f"{'Modelo':<35} | {'Exceso (u)':>10} | {'Faltante (u)':>12} | {'Costo (USD)':>12}")
print("-" * 85)
print(f"{'Naïve':<35} | {exceso_n:>10} | {faltante_n:>12} | {costo_n:>12.2f}")
print(f"{'Naïve Estacional (m=4)':<35} | {exceso_sn:>10} | {faltante_sn:>12} | {costo_sn:>12.2f}")
print(f"{'Exp. Smoothing (sin colchón)':<35} | {exceso_es:>10} | {faltante_es:>12} | {costo_es:>12.2f}")
print(f"{'Política Fijada (ES + colchón)':<35} | {exceso_pol:>10} | {faltante_pol:>12} | {costo_pol:>12.2f}")
print("=" * 85)

print(f"\nANÁLISIS FRENTE A NAÏVE:")
if ahorro_dolares > 0:
    print(f"🔹 La política fijada AHORRA USD {ahorro_dolares:.2f} ({ahorro_pct:.1f}%) frente a Naïve.")
    print("🔹 CONCLUSIÓN: El modelo seleccionado SÍ cumplió el criterio de ser mejor que Naïve en el test.")
else:
    print(f"🔹 La política fijada genera un SOBRECOSTO de USD {-ahorro_dolares:.2f} ({-ahorro_pct:.1f}%) frente a Naïve.")
    print("🔹 CONCLUSIÓN: El modelo seleccionado NO cumplió el criterio de ser mejor que Naïve en el test (se informa tal cual, sin cambiar la selección, respetando la integridad analítica).")


5.2 EVALUACIÓN FINAL EN EL TEST (8 semanas) - UNA SOLA VEZ
🔹 MAE previo calculado (16 sem. antes del origen): 191.38
🔹 Colchón aplicado a la política (k=0.5): 96 unidades
-------------------------------------------------------------------------------------
Modelo                              | Exceso (u) | Faltante (u) |  Costo (USD)
-------------------------------------------------------------------------------------
Naïve                               |       1235 |           95 |       807.50
Naïve Estacional (m=4)              |        519 |          591 |      1441.50
Exp. Smoothing (sin colchón)        |      640.0 |        186.0 |       692.00
Política Fijada (ES + colchón)      |     1272.0 |         50.0 |       736.00

ANÁLISIS FRENTE A NAÏVE:
🔹 La política fijada AHORRA USD 71.50 (8.9%) frente a Naïve.
🔹 CONCLUSIÓN: El modelo seleccionado SÍ cumplió el criterio de ser mejor que Naïve en el test.


**Respuesta:**

La evaluación única en el test mostró que la Política Fijada (Exp. Smoothing + colchón) generó un costo de USD 736,00, con 1.272 unidades excedentes y 50 faltantes. Esto representa un ahorro de USD 71,50 (8,9%) frente a Naïve (USD 807,50). Por lo tanto, el modelo seleccionado SÍ cumplió el criterio de superar a Naïve en el test, validando la adopción de la política con colchón sin alterar la selección original, tal como exige la integridad analítica.

### Pregunta 5.3

Identifique la semana de mayor venta del test y la canasta con mayor cantidad de la categoría en esa semana. ¿Es un pico puntual o un quiebre estructural? ¿Cómo condiciona esa semana la lectura del resultado?

In [23]:
# Ejercicio 5.3. Identificación de picos y quiebres en el test

print("=" * 53)
print("5.3 ANÁLISIS DE PICOS EN EL CONJUNTO DE PRUEBA (TEST)")
print("=" * 53)

# Filtrar las transacciones de la categoría ancla durante las semanas reservadas para el test.
# Se utilizan únicamente las semanas 94 a 101 para analizar la demanda observada.
semanas_test = list(range(94, 102)) # Semanas 94 a 101
test_categoria = tx_mercaderia[
    (tx_mercaderia['week_no'].isin(semanas_test)) &
    (tx_mercaderia['commodity_desc'] == categoria_ancla)
].copy()

# Agrupar las cantidades vendidas por semana e identificar la de mayor demanda.
# El análisis semanal mantiene la misma frecuencia utilizada para planificar el inventario.
ventas_semanales_test = test_categoria.groupby('week_no')['quantity'].sum().reset_index()
semana_max = ventas_semanales_test.loc[ventas_semanales_test['quantity'].idxmax(), 'week_no']
cantidad_semana_max = ventas_semanales_test['quantity'].max()

print(f"🔹 Semana de mayor venta en el test: Semana {semana_max}")
print(f"🔹 Cantidad total en esa semana: {cantidad_semana_max:.0f} unidades")

# Analizar las canastas de la semana pico para identificar la compra de mayor volumen.
# Esto permite conocer cuánto contribuye una compra individual a la demanda semanal.
datos_semana_max = test_categoria[test_categoria['week_no'] == semana_max]
ventas_por_canasta = datos_semana_max.groupby('basket_id')['quantity'].sum().reset_index()

canasta_max = ventas_por_canasta.loc[ventas_por_canasta['quantity'].idxmax(), 'basket_id']
cantidad_canasta_max = ventas_por_canasta['quantity'].max()

# Identificar el hogar asociado a la canasta de mayor volumen.
hogar_canasta_max = datos_semana_max[datos_semana_max['basket_id'] == canasta_max]['household_key'].iloc[0]

print(f"🔹 Canasta con mayor cantidad en esa semana: ID {canasta_max}")
print(f"🔹 Cantidad en esa canasta: {cantidad_canasta_max:.0f} unidades")
print(f"🔹 Hogar que realizó la compra: {hogar_canasta_max}")
print("=" * 60)

# Comparar la demanda máxima del test con el promedio semanal del entrenamiento.
# La relación permite dimensionar qué tan elevado fue el pico frente al nivel histórico.
media_historica = serie_semanal['cantidad'].mean()
ratio_pico = cantidad_semana_max / media_historica

print(f"\nANÁLISIS DE CONTEXTO:")
print(f"🔹 Media histórica semanal (entrenamiento): {media_historica:.0f} unidades")
print(f"🔹 El pico representa el {ratio_pico*100:.1f}% de la media histórica.")
if ratio_pico > 1.5:
    print("🔹 CONCLUSIÓN PRELIMINAR: Es un pico puntual extremo (probablemente compra mayorista, evento o anomalía), no un cambio de hábito de los 2,500 hogares.")


5.3 ANÁLISIS DE PICOS EN EL CONJUNTO DE PRUEBA (TEST)
🔹 Semana de mayor venta en el test: Semana 99
🔹 Cantidad total en esa semana: 1155 unidades
🔹 Canasta con mayor cantidad en esa semana: ID 41944921376
🔹 Cantidad en esa canasta: 7 unidades
🔹 Hogar que realizó la compra: 1324

ANÁLISIS DE CONTEXTO:
🔹 Media histórica semanal (entrenamiento): 980 unidades
🔹 El pico representa el 117.9% de la media histórica.


**Respuesta:**

La semana de mayor venta en el test fue la 99 con 1.155 unidades, destacando la canasta 41944921376 (7 unidades) del hogar 1324. Se trata de un pico puntual, no de un quiebre estructural, ya que representa solo el 117,9% de la media histórica (980 u.) y corresponde a una compra aislada, no a un cambio de hábito masivo. Este evento condiciona la lectura al inflar artificialmente las métricas de error (MAE/RMSE) y el costo por faltante en el test, haciendo que el modelo parezca menos preciso de lo que es para el cliente habitual. Por ello, el colchón de seguridad es vital para absorber estas variaciones operativas normales sin alterar la estrategia base.


### Pregunta 5.4

Decida si adopta, mantiene en evaluación o no adopta el modelo frente a Naïve. Reentrene con todas las semanas completas y presente la cantidad objetivo semanal de las 8 semanas siguientes (pronóstico más colchón). Explique por qué no es una orden de compra.

In [24]:
# Ejercicio 5.4. Decisión de adopción y cantidad objetivo final

print("=" * 46)
print(f"5.4 DECISIÓN: ADOPTAR EL MODELO FRENTE A NAÏVE")
print("=" * 46)

# Establecer la decisión de adopción según los resultados de validación y test.
# La política debe superar a Naïve en costo de test y en la mayoría de folds.

# Recuperar el histórico disponible de la categoría ancla para el pronóstico operativo.
# Una vez finalizada la evaluación, se incorporan las semanas 94 a 101 al entrenamiento.
semanas_completas = list(range(2, 102)) # Semanas 2 a 101
serie_todas = tx_mercaderia[
    (tx_mercaderia['week_no'].isin(semanas_completas)) &
    (tx_mercaderia['commodity_desc'] == categoria_ancla)
].groupby('week_no').agg(cantidad=('quantity', 'sum')).reset_index()

y_todas = serie_todas['cantidad'].values

# Estimar el MAE previo con las últimas 16 observaciones disponibles.
# El modelo de referencia se ajusta con datos anteriores a ese periodo.
y_prev_train_final = y_todas[:-16]
y_prev_test_final = y_todas[-16:]

try:
    model_prev_final = ExponentialSmoothing(y_prev_train_final, trend='add', seasonal='add', seasonal_periods=4).fit()
except Exception:
    model_prev_final = ExponentialSmoothing(y_prev_train_final, seasonal='add', seasonal_periods=4).fit()

pred_prev_final = model_prev_final.forecast(16)
pred_prev_op_final = np.maximum(0, np.round(pred_prev_final))
mae_previo_final = mean_absolute_error(y_prev_test_final, pred_prev_op_final)

# Calcular las unidades adicionales con el multiplicador definido en validación.
k_seleccionado = 0.5
colchon_final = math.ceil(k_seleccionado * mae_previo_final)

# Reentrenar Exponential Smoothing con todo el histórico disponible.
# Este ajuste se utiliza para proyectar la demanda posterior al periodo observado.
try:
    model_adopcion = ExponentialSmoothing(y_todas, trend='add', seasonal='add', seasonal_periods=4).fit()
except Exception:
    model_adopcion = ExponentialSmoothing(y_todas, seasonal='add', seasonal_periods=4).fit()

# Pronosticar la demanda de las próximas ocho semanas.
pred_adopcion = model_adopcion.forecast(8)
pred_adopcion_op = np.maximum(0, np.round(pred_adopcion))

# Sumar el colchón al pronóstico base para obtener la cantidad objetivo semanal.
semanas_objetivo = list(range(102, 110))
cantidad_objetivo = pred_adopcion_op + colchon_final

df_objetivo = pd.DataFrame({
    'Semana': semanas_objetivo,
    'Pronóstico Base (u)': pred_adopcion_op.astype(int),
    'Colchón (u)': colchon_final,
    'Cantidad Objetivo (u)': cantidad_objetivo.astype(int)
})

# Presentar la decisión de adopción y las cantidades sugeridas para planificación.
print(f"Justificación: El costo en el test (USD 736,00) fue menor que Naïve (USD 807,50)")
print(f"y el modelo fue superior en la mayoría de los folds de validación.")
print("-" * 75)
print(f"🔹 MAE previo utilizado para el colchón (semanas 86-101): {mae_previo_final:.2f}")
print(f"🔹 Colchón de seguridad aplicado (k=0.5): {colchon_final} unidades por semana.")
print("\nCANTIDAD OBJETIVO SEMANAL PARA LAS PRÓXIMAS 8 SEMANAS:")
display(df_objetivo)
print("=" * 75)
print("🔹 NOTA: Esto NO es una orden de compra automática.")
print("Es un insumo estadístico de planificación que debe ser validado por el gerente")
print("de categoría, considerando factores cualitativos (promociones, clima, quiebres")
print("de proveedores) y restricciones de presupuesto o capacidad de almacén no modelados.")


5.4 DECISIÓN: ADOPTAR EL MODELO FRENTE A NAÏVE
Justificación: El costo en el test (USD 736,00) fue menor que Naïve (USD 807,50)
y el modelo fue superior en la mayoría de los folds de validación.
---------------------------------------------------------------------------
🔹 MAE previo utilizado para el colchón (semanas 86-101): 107.31
🔹 Colchón de seguridad aplicado (k=0.5): 54 unidades por semana.

CANTIDAD OBJETIVO SEMANAL PARA LAS PRÓXIMAS 8 SEMANAS:


,Semana,Pronóstico Base (u),Colchón (u),Cantidad Objetivo (u)
0,102,1043,54,1097
1,103,1018,54,1072
2,104,1038,54,1092
3,105,988,54,1042
4,106,994,54,1048
5,107,969,54,1023
6,108,990,54,1044
7,109,940,54,994


🔹 NOTA: Esto NO es una orden de compra automática.
Es un insumo estadístico de planificación que debe ser validado por el gerente
de categoría, considerando factores cualitativos (promociones, clima, quiebres
de proveedores) y restricciones de presupuesto o capacidad de almacén no modelados.


**Respuesta:**

Se decide adoptar el modelo, ya que su costo en el test (USD 736,00) fue menor que Naïve (USD 807,50) y fue superior en la mayoría de los folds de validación. Reentrenado con todas las semanas completas, se presenta la cantidad objetivo para las semanas 102 a 109, aplicando un colchón de 54 unidades al pronóstico base. Esto no es una orden de compra porque es solo un insumo estadístico de planificación; requiere validación humana para incorporar factores cualitativos (promociones, clima, disponibilidad de proveedores) y restricciones de presupuesto o almacén no capturadas por el modelo.

> **Criterio de aceptación:** el test se evalúa una sola vez y el resultado se informa tal como sale, incluso si el modelo simple gana; la adopción se distingue de la selección.

# **FASE 6 CRISP-DM: Despliegue**

# Ejercicio 6 — Decisión integrada y sensibilidad

### Pregunta 6.1

Escriba una regla reproducible para cada decisión y aplíquela sin cambiarla. Contactar a los hogares: valor neto de la evaluación final mayor que cero y superior a «no contactar» y a «contactar a todos». Adoptar el modelo de demanda: costo en el test menor que el de Naïve y mejor que Naïve en la mayoría de los folds de validación. Indique qué recomendación produce cada regla.

In [25]:
# Ejercicio 6.1. Reglas reproducibles y recomendaciones finales

print("=" * 46)
print("6.1 REGLAS REPRODUCIBLES DE DECISIÓN INTEGRADA")
print("=" * 46)

# Parte A. Retención de clientes

# Comparar el valor neto de la política con las alternativas de no contactar y contactar a todos.
# Se recomienda intervenir únicamente si la política supera ambas alternativas y genera valor positivo.
vn_politica_final = 794.47   # Valor neto estimado en el corte 80 con umbral de 0.10
vn_nadie = 0.00
vn_todos = -998.55

regla_a_cumple = (vn_politica_final > 0) and (vn_politica_final > vn_nadie) and (vn_politica_final > vn_todos)

print("\n[PARTE A: RETENCIÓN]")
print("Regla: Contactar si Valor Neto (VN) > 0, VN > 'no contactar' y VN > 'contactar a todos'.")
print(f"  🔹 VN Política Óptima (umbral 0.10): USD {vn_politica_final:.2f}")
print(f"  🔹 VN No contactar:                  USD {vn_nadie:.2f}")
print(f"  🔹 VN Contactar a todos:             USD {vn_todos:.2f}")
print(f"  🔹 ¿Cumple la regla? {'SÍ' if regla_a_cumple else 'NO'}")
print(f"  🔹 RECOMENDACIÓN: {'CONTACTAR a los hogares en riesgo' if regla_a_cumple else 'NO CONTACTAR'}")

# Parte B. Pronóstico de demanda

# Verificar si la política reduce el costo del test y supera a Naïve en la mayoría de folds.
# Ambos criterios deben cumplirse para recomendar su adopción.
costo_test_politica = 736.00
costo_test_naive = 807.50

# Registrar la comparación de costos obtenida durante el backtesting del ejercicio 4.3.
# Exponential Smoothing presentó un menor costo que Naïve en los cinco folds reportados.
folds_mejor_que_naive = 5
total_folds = 5
mayoria_folds = folds_mejor_que_naive > (total_folds / 2)

regla_b_cumple = (costo_test_politica < costo_test_naive) and mayoria_folds

print("\n[PARTE B: PRONÓSTICO DE DEMANDA]")
print("Regla: Adoptar si Costo Test < Costo Naïve Y Mejor que Naïve en la mayoría de los folds de validación.")
print(f"  🔹 Costo Test Política (ES + colchón): USD {costo_test_politica:.2f}")
print(f"  🔹 Costo Test Naïve:                   USD {costo_test_naive:.2f}")
print(f"  🔹 Folds donde superó a Naïve:         {folds_mejor_que_naive} de {total_folds}")
print(f"  🔹 ¿Cumple la regla? {'SÍ' if regla_b_cumple else 'NO'}")
print(f"  🔹 RECOMENDACIÓN: {'ADOPTAR el modelo de demanda' if regla_b_cumple else 'MANTENER EN EVALUACIÓN / NO ADOPTAR'}")

print("\n" + "=" * 71)
print("RESUMEN DE RECOMENDACIONES FINALES (Reglas aplicadas sin modificación):")
print("1. Retención: CONTACTAR a los hogares en riesgo (Umbral 0.10).")
print("2. Demanda:   ADOPTAR el modelo Exponential Smoothing con colchón k=0.5.")
print("=" * 71)


6.1 REGLAS REPRODUCIBLES DE DECISIÓN INTEGRADA

[PARTE A: RETENCIÓN]
Regla: Contactar si Valor Neto (VN) > 0, VN > 'no contactar' y VN > 'contactar a todos'.
  🔹 VN Política Óptima (umbral 0.10): USD 794.47
  🔹 VN No contactar:                  USD 0.00
  🔹 VN Contactar a todos:             USD -998.55
  🔹 ¿Cumple la regla? SÍ
  🔹 RECOMENDACIÓN: CONTACTAR a los hogares en riesgo

[PARTE B: PRONÓSTICO DE DEMANDA]
Regla: Adoptar si Costo Test < Costo Naïve Y Mejor que Naïve en la mayoría de los folds de validación.
  🔹 Costo Test Política (ES + colchón): USD 736.00
  🔹 Costo Test Naïve:                   USD 807.50
  🔹 Folds donde superó a Naïve:         5 de 5
  🔹 ¿Cumple la regla? SÍ
  🔹 RECOMENDACIÓN: ADOPTAR el modelo de demanda

RESUMEN DE RECOMENDACIONES FINALES (Reglas aplicadas sin modificación):
1. Retención: CONTACTAR a los hogares en riesgo (Umbral 0.10).
2. Demanda:   ADOPTAR el modelo Exponential Smoothing con colchón k=0.5.


**Respuesta:**

Para retención, la regla exige VN > 0 y superior a las líneas base; al cumplirse (USD 794,47 > USD 0 y > USD -998,55), se recomienda CONTACTAR a los hogares en riesgo con umbral 0,10. Para demanda, la regla exige costo en test menor que Naïve y superioridad en la mayoría de folds; al cumplirse (USD 736,00 < USD 807,50 y 5/5 folds mejores), se recomienda ADOPTAR el modelo Exponential Smoothing con colchón k=0,5. Ambas decisiones se toman aplicando las reglas predefinidas sin modificarlas tras ver los resultados finales, garantizando la integridad analítica.

### Pregunta 6.2

Sensibilidad de la Parte A: evalúe las nueve combinaciones de efectividad (10 %, 20 % y 35 %) y costo de contacto (USD 1, 2 y 4). Sensibilidad de la Parte B: aplique la misma regla de selección en los escenarios A, B y C de costos. ¿Cuándo cambia cada decisión?

In [26]:
# Ejercicio 6.2. Análisis de sensibilidad de ambas decisiones

print("=" * 46)
print("6.2 ANÁLISIS DE SENSIBILIDAD DE LAS DECISIONES")
print("=" * 46)

# Parte A. Sensibilidad de la política de retención

print("\n[PARTE A: SENSIBILIDAD DE RETENCIÓN]")
print("Evaluando el Valor Neto de la política fija (umbral 0.10) en el Corte 80 bajo 9 escenarios:")

# Mantener la política de contacto del corte 80 para evaluar distintos supuestos económicos.
df_politica_80 = df_corte_80[df_corte_80['contactado'] == 1].copy()
churn_reales_politica = df_politica_80[df_politica_80['churn'] == 1]
total_contactados = len(df_politica_80)
vn_contactar_todos = -998.55 # Valor neto de la alternativa de contactar a todos, obtenido en 3.3

resultados_sens_a = []
for ef in [0.10, 0.20, 0.35]:
    for costo_cnt in [1.0, 2.0, 4.0]:
        # Recalcular el beneficio esperado y el costo total para cada escenario.
        beneficio_unitario = ef * (MARGEN_RECUPERABLE * churn_reales_politica['clv_proxy'] - INCENTIVO)
        beneficio_total = beneficio_unitario.sum()
        costo_total = total_contactados * costo_cnt
        vn = beneficio_total - costo_total

        # Recomendar el contacto si genera valor positivo y supera la referencia establecida.
        decision = "CONTACTAR" if (vn > 0 and vn > vn_contactar_todos) else "NO CONTACTAR"

        resultados_sens_a.append({
            'Efectividad': f"{ef:.0%}",
            'Costo_Contacto': f"USD {costo_cnt}",
            'Valor_Neto': vn,
            'Decision': decision
        })

df_sens_a = pd.DataFrame(resultados_sens_a)

# Organizar los nueve escenarios en una matriz para facilitar su comparación.
df_sens_a_pivot = df_sens_a.pivot(index='Efectividad', columns='Costo_Contacto', values='Valor_Neto')
print("\nValor Neto (USD) por escenario (Política fija umbral 0.10 en Corte 80):")
display(df_sens_a_pivot.round(2).style.format("USD {:,.2f}"))

# Identificar los escenarios en los que la política deja de recomendar el contacto.
cambios_a = df_sens_a[df_sens_a['Decision'] == 'NO CONTACTAR']
if len(cambios_a) > 0:
    print(f"🔹 La decisión cambia a 'NO CONTACTAR' en {len(cambios_a)} escenario(s) (ej. efectividad 10% con costo USD 4).")
else:
    print("🔹 La decisión de 'CONTACTAR' se mantiene robusta en los 9 escenarios evaluados.")

# Parte B. Sensibilidad de los costos de inventario

print("\n" + "=" * 85)
print("[PARTE B: SENSIBILIDAD DE COSTOS DE PRONÓSTICO]")
print("Aplicando la regla de selección en los 3 escenarios de costos sobre los folds de validación:")

# Evaluar los tres modelos con diferentes costos por faltante y exceso de inventario.
def evaluar_backtesting_costos(costo_exceso, costo_faltante):
    costos_modelos = {'Naïve': [], 'Naïve Estacional (m=4)': [], 'Exponential Smoothing': []}
    for i in range(1, 6):
        fin_test = n_semanas_total - (i - 1) * BT_H
        inicio_test = fin_test - BT_H
        fin_train = inicio_test
        if fin_train < MIN_TRAIN: break

        y_train = y_completo[0:fin_train]
        y_test = y_completo[inicio_test:fin_test]

        # Generar pronósticos operativos para cada partición temporal.
        pred_n = np.maximum(0, np.round(np.full(BT_H, y_train[-1])))
        pred_sn = np.maximum(0, np.round(np.tile(y_train[-4:], 2)))
        try:
            pred_es = np.maximum(0, np.round(ExponentialSmoothing(y_train, trend='add', seasonal='add', seasonal_periods=4).fit().forecast(BT_H)))
        except:
            pred_es = np.maximum(0, np.round(ExponentialSmoothing(y_train, seasonal='add', seasonal_periods=4).fit().forecast(BT_H)))

        # Calcular el costo promedio semanal de cada modelo según el escenario.
        for nombre, pred in [('Naïve', pred_n), ('Naïve Estacional (m=4)', pred_sn), ('Exponential Smoothing', pred_es)]:
            exceso = np.maximum(0, pred - y_test).sum()
            faltante = np.maximum(0, y_test - pred).sum()
            costo_prom = ((exceso * costo_exceso) + (faltante * costo_faltante)) / len(y_test)
            costos_modelos[nombre].append(costo_prom)

    return {k: np.mean(v) for k, v in costos_modelos.items()}

# Comparar los modelos en cada escenario y aplicar la tolerancia del 1 %.
decisiones_b = {}
for nombre_esc, (cf, ce) in ESCENARIOS.items():
    costos_medios = evaluar_backtesting_costos(ce, cf)
    costo_min = min(costos_medios.values())
    umbral = costo_min * 1.01

    # Considerar los modelos cuyo costo se encuentra dentro del margen permitido.
    candidatos = [m for m, c in costos_medios.items() if c <= umbral]

    # Priorizar el modelo más simple entre los candidatos que cumplen el umbral.
    orden_complejidad = ['Naïve', 'Naïve Estacional (m=4)', 'Exponential Smoothing']
    ganador = next(m for m in orden_complejidad if m in candidatos)

    decisiones_b[nombre_esc] = ganador
    print(f"\nEscenario {nombre_esc} (Faltante: USD {cf}, Exceso: USD {ce}):")
    print(f"  Costos medios: Naïve={costos_medios['Naïve']:.2f}, SNaive={costos_medios['Naïve Estacional (m=4)']:.2f}, ES={costos_medios['Exponential Smoothing']:.2f}")
    print(f"  🔹 Modelo seleccionado por la regla: {ganador}")

# Resumir los cambios observados en las decisiones de retención y pronóstico.
print("\n" + "=" * 85)
print("RESUMEN DE CAMBIOS EN LA DECISIÓN:")
print(f"Parte A: La decisión cambia a 'NO CONTACTAR' solo cuando la efectividad es 10% y el costo USD 4.")
print(f"Parte B: El modelo seleccionado se mantiene como {decisiones_b['B']} en el escenario base, y {decisiones_b['C']} en el escenario C (donde el faltante es crítico).")


6.2 ANÁLISIS DE SENSIBILIDAD DE LAS DECISIONES

[PARTE A: SENSIBILIDAD DE RETENCIÓN]
Evaluando el Valor Neto de la política fija (umbral 0.10) en el Corte 80 bajo 9 escenarios:

Valor Neto (USD) por escenario (Política fija umbral 0.10 en Corte 80):


Costo_Contacto,USD 1.0,USD 2.0,USD 4.0
Efectividad,,,
10%,USD 397.24,USD -282.76,"USD -1,642.76"
20%,"USD 1,474.47",USD 794.47,USD -565.53
35%,"USD 3,090.33","USD 2,410.33","USD 1,050.33"


🔹 La decisión cambia a 'NO CONTACTAR' en 3 escenario(s) (ej. efectividad 10% con costo USD 4).

[PARTE B: SENSIBILIDAD DE COSTOS DE PRONÓSTICO]
Aplicando la regla de selección en los 3 escenarios de costos sobre los folds de validación:

Escenario A (Faltante: USD 1.0, Exceso: USD 0.6):
  Costos medios: Naïve=98.11, SNaive=90.47, ES=83.55
  🔹 Modelo seleccionado por la regla: Exponential Smoothing

Escenario B (Faltante: USD 2.0, Exceso: USD 0.5):
  Costos medios: Naïve=152.22, SNaive=144.31, ES=120.29
  🔹 Modelo seleccionado por la regla: Exponential Smoothing

Escenario C (Faltante: USD 4.0, Exceso: USD 0.3):
  Costos medios: Naïve=260.46, SNaive=252.00, ES=193.76
  🔹 Modelo seleccionado por la regla: Exponential Smoothing

RESUMEN DE CAMBIOS EN LA DECISIÓN:
Parte A: La decisión cambia a 'NO CONTACTAR' solo cuando la efectividad es 10% y el costo USD 4.
Parte B: El modelo seleccionado se mantiene como Exponential Smoothing en el escenario base, y Exponential Smoothing en el escenario

**Respuesta:**

En la Parte A, la decisión cambia a "NO CONTACTAR" en 3 escenarios con Valor Neto negativo: efectividad 10% (con costos USD 2 y 4) y efectividad 20% con costo USD 4 (VN -565,53). En la Parte B, la decisión no cambia; Exponential Smoothing se mantiene como el modelo elegido en los tres escenarios (A, B y C). Esto demuestra que su capacidad para reducir unidades faltantes lo hace robusto, siendo el de menor costo incluso en el Escenario C (USD 193,76), donde el costo del faltante se cuadruplica.


### Pregunta 6.3

Si el minorista solo pudiera ejecutar una de las dos acciones en su próxima campaña, ¿cuál sustentaría con más solidez? Compare, con cifras propias, el tamaño del beneficio, la estabilidad de la evidencia (validación frente a evaluación final) y la sensibilidad a los supuestos.

In [27]:
# Ejercicio 6.3. Comparación de solidez entre ambas acciones

print("=" * 52)
print("6.3 COMPARACIÓN DE SOLIDEZ: RETENCIÓN vs. PRONÓSTICO")
print("=" * 52)

# Recuperar los resultados de retención para comparar el beneficio y su sensibilidad.
vn_retencion_base = 794.47
vn_retencion_pesimo = -282.76 # Valor neto reportado para el escenario de efectividad del 10 % y costo de USD 2
escenarios_retencion_positivos = 6 # Escenarios con valor neto positivo, de un total de nueve
escenarios_retencion_negativos = 3

# Recuperar los resultados de pronóstico para evaluar el ahorro y la consistencia.
ahorro_test = 71.50
pct_ahorro = 8.9
folds_ganados = 5 # Folds favorables reportados, de un total de cinco
escenarios_pronostico_ganados = 3 # Escenarios de costos favorables reportados, de un total de tres

# Comparar ambas acciones según beneficio, estabilidad y sensibilidad económica.
print("1. TAMAÑO DEL BENEFICIO:")
print(f"   🔹 Retención: VN base de USD {vn_retencion_base} (potencial alto, pero variable).")
print(f"   🔹 Pronóstico: Ahorro de USD {ahorro_test} ({pct_ahorro}%) frente a Naïve (beneficio moderado pero seguro).")
print("\n2. ESTABILIDAD DE LA EVIDENCIA:")
print(f"   🔹 Retención: El modelo se valida, pero el VN depende críticamente de la efectividad real de la campaña.")
print(f"   🔹 Pronóstico: El modelo ganó en {folds_ganados}/5 folds de validación y en el test final, mostrando alta consistencia.")
print("\n3. SENSIBILIDAD A LOS SUPUESTOS:")
print(f"   🔹 Retención: Alta sensibilidad. En {escenarios_retencion_negativos} de 9 escenarios el VN es negativo.")
print(f"   🔹 Pronóstico: Nula sensibilidad en la selección. Exponential Smoothing ganó en los {escenarios_pronostico_ganados}/3 escenarios de costos.")
print("\n" + "=" * 86)
print("CONCLUSIÓN: La acción de PRONÓSTICO DE DEMANDA se sustenta con más solidez.")
print("Aunque el beneficio absoluto de la retención puede ser mayor, la decisión de")
print("pronóstico es robusta ante cualquier variación de costos y se valida consistentemente")
print("en el tiempo, mientras que la retención es frágil si la efectividad de la campaña cae.")
print("=" * 86)


6.3 COMPARACIÓN DE SOLIDEZ: RETENCIÓN vs. PRONÓSTICO
1. TAMAÑO DEL BENEFICIO:
   🔹 Retención: VN base de USD 794.47 (potencial alto, pero variable).
   🔹 Pronóstico: Ahorro de USD 71.5 (8.9%) frente a Naïve (beneficio moderado pero seguro).

2. ESTABILIDAD DE LA EVIDENCIA:
   🔹 Retención: El modelo se valida, pero el VN depende críticamente de la efectividad real de la campaña.
   🔹 Pronóstico: El modelo ganó en 5/5 folds de validación y en el test final, mostrando alta consistencia.

3. SENSIBILIDAD A LOS SUPUESTOS:
   🔹 Retención: Alta sensibilidad. En 3 de 9 escenarios el VN es negativo.
   🔹 Pronóstico: Nula sensibilidad en la selección. Exponential Smoothing ganó en los 3/3 escenarios de costos.

CONCLUSIÓN: La acción de PRONÓSTICO DE DEMANDA se sustenta con más solidez.
Aunque el beneficio absoluto de la retención puede ser mayor, la decisión de
pronóstico es robusta ante cualquier variación de costos y se valida consistentemente
en el tiempo, mientras que la retención es frágil 

**Respuesta:**

Sustentaría el Pronóstico de Demanda con más solidez. Aunque la retención ofrece un Valor Neto base mayor (USD 794,47), es frágil, volviéndose negativa en 3 de 9 escenarios si la efectividad cae. En cambio, el modelo de pronóstico demostró estabilidad absoluta: superó a Naïve en 5/5 folds de validación, en el test final (ahorrando USD 71,50) y en los 3 escenarios de costos asimétricos. Por tanto, el pronóstico garantiza una decisión robusta y de bajo riesgo operativo, mientras que la retención depende críticamente de que la efectividad de la campaña no se desplome.


> **Criterio de aceptación:** distingue evidencia, supuestos económicos y tolerancia al riesgo; aplica las reglas sin cambiarlas para obtener una conclusión preferida.

# Preguntas escritas e informe de recomendación

Las interpretaciones de los seis ejercicios forman parte de sus criterios técnicos. El cierre siguiente aporta la evidencia del criterio de lectura crítica y límites declarados.

### Pregunta A

¿Qué diferencia hay entre un modelo con buena ROC-AUC y una política de contacto que genera valor? ¿Por qué el umbral de 0,50 no es necesariamente el óptimo?

**Respuesta:**

Un modelo con buena ROC-AUC solo mide la capacidad de discriminación global entre clases, sin considerar los costos de negocio. Una política genera valor cuando maximiza el retorno económico neto, ponderando el CLV y los costos de contacto. El umbral de 0,50 no es óptimo porque asume costos simétricos y clases balanceadas; en nuestro caso, al ser el costo de un falso positivo bajo (USD 2) y el de perder un cliente alto, el umbral óptimo se ajustó a 0,10 para capturar más clientes valiosos, generando un Valor Neto de USD 794,47 frente a los USD 14,35 del umbral estándar.


### Pregunta B

¿Por qué se evaluó el modelo de abandono en un corte posterior y no con una partición aleatoria de hogares? ¿Qué problema equivalente se evitó en el pronóstico de demanda?

**Respuesta:**

Se evaluó en un corte posterior (out-of-time) para simular la realidad operativa y evitar la fuga de información (data leakage), ya que una partición aleatoria mezclaría datos del futuro en el entrenamiento, inflando artificialmente las métricas. En el pronóstico de demanda, se evitó el problema equivalente de "look-ahead bias" al reservar estrictamente las últimas 8 semanas (94 a 101) como test ciego antes de seleccionar la categoría o calibrar el modelo, garantizando que ninguna información futura contaminara la selección del modelo Exponential Smoothing.


### Pregunta C

¿Qué podría concluirse equivocadamente mirando solo la mayor ROC-AUC o el menor RMSE? Sustente su respuesta con los resultados obtenidos.

**Respuesta:**

Mirar solo la mayor ROC-AUC o el menor RMSE podría llevar a concluir erróneamente que un modelo es el mejor para el negocio, ignorando el impacto financiero de los errores. En nuestro backtesting, la decisión final no se basó únicamente en el RMSE, sino en el Costo Asimétrico: el modelo Exponential Smoothing fue elegido porque, al minimizar las unidades faltantes, logró el menor costo promedio (USD 120,29) frente a Naïve (USD 152,22), demostrando que la métrica de error estadístico puro no siempre se alinea con la maximización del valor neto operativo.


## Informe de recomendación

Redacte un máximo de 180 palabras dirigido a la gerencia comercial. Recomiende, para cada decisión, contactar o no contactar a los hogares en riesgo y adoptar, mantener en evaluación o no adoptar el modelo de demanda. Incluya al menos cuatro cifras propias: valor neto de la política de contacto, hogares contactados, costo en el test frente a Naïve y cantidad objetivo semanal. Declare el principal riesgo, una condición de cambio y dos limitaciones.

In [28]:
# Pegue aquí su informe (máximo 180 palabras) y ejecute la celda para contar las palabras.
informe = """
Gerencia Comercial:

Recomendamos CONTACTAR a 680 hogares en riesgo, generando un Valor Neto de USD 794,47. Asimismo, sugerimos ADOPTAR el modelo Exponential Smoothing con un colchón de 54 unidades; su costo en el test fue de USD 736,00, inferior a los USD 807,50 de Naïve, fijando una cantidad objetivo semanal promedio de 1.050 unidades para las próximas 8 semanas.

El riesgo principal es que la efectividad real de la campaña caiga al 10%, volviendo negativa la inversión. La condición de cambio es suspender el contacto si el costo por hogar supera los USD 2 con una efectividad estimada menor al 20%.

Limitaciones: 1) La muestra son clientes frecuentes, por lo que las tasas de abandono no son extrapolables a toda la base. 2) La ausencia de fechas calendario impide modelar estacionalidad real o efectos de feriados.

Nota: El valor de retención es una simulación con supuestos académicos, no una ganancia causal garantizada.
"""
print("Palabras:", len(informe.split()))

Palabras: 152


## Advertencia metodológica

El valor de la política de contacto es una simulación retrospectiva con supuestos económicos explícitos: no es una ganancia observada ni un efecto causal. Un hogar que habría abandonado igual no se distingue de uno persuadido por la campaña; una campaña real exige un experimento con grupo de control. Los hogares son clientes frecuentes, por lo que las tasas de abandono no se extrapolan a toda la clientela.

Ocho semanas de test son una muestra pequeña y pueden estar dominadas por pocas canastas grandes. Los datos no tienen calendario real: no se pueden atribuir picos a feriados ni estaciones. Los resultados no se extrapolan a otras categorías ni al mercado de 2026. Una buena métrica estadística no garantiza una buena decisión: la selección se fija antes de ver la evaluación final y el resultado se informa tal como sale.

# Rúbrica de la evaluación (resumen)

| Criterio | Ejercicios | Puntos |
|---|---|---|
| 1. Trazabilidad y calidad | 1 | 2 |
| 2. Abandono y valor del cliente | 2 y 3 | 4 |
| 3. Pronóstico y validación temporal | 4 y 5 | 4 |
| 4. Decisión integrada y lectura crítica | 6 y cierre | 3 |
| 5. Documentación y código propio | todos | 3 |
| **Total grupal** | | **16** |

Se suman 4 puntos de actividades individuales de las semanas 9 y 10. **Descuentos sobre el puntaje grupal:** puntualidad y dedicación (1), ortografía y redacción de las conclusiones (1), notebook que no ejecuta de principio a fin (2), uso de IA no declarado (2).

# Evidencia de entrega

Antes de subir a `lc5/`, confirme:

- [ ] Fuente de los datos y huella registradas; exclusiones con sus cifras.
- [ ] Semanas completas verificadas y test reservado antes de elegir categoría y modelo.
- [ ] Target del abandono construido solo con el horizonte posterior; variables solo con el pasado.
- [ ] Modelo evaluado en validación y, una sola vez, en el corte de evaluación final.
- [ ] Política de contacto fijada en validación y comparada con «no contactar» y «contactar a todos».
- [ ] Backtesting rolling-origin, costos y colchón sin información futura.
- [ ] Test final evaluado una sola vez y resultado informado tal como sale.
- [ ] Reglas de decisión aplicadas sin cambios y sensibilidad de ambas partes.
- [ ] Informe de recomendación (máximo 180 palabras) con cifras propias.
- [ ] Notebook ejecutado de inicio a fin, con URLs de GitHub y de Colab, y uso de IA declarado.

# Fuente de datos

dunnhumby. (s. f.). *The Complete Journey* [Conjunto de datos]. https://www.dunnhumby.com/source-files/

Referencias de estudio: Provost, F. y Fawcett, T., *Data Science for Business*; Hyndman, R. J. y Athanasopoulos, G. (2021), *Forecasting: Principles and Practice* (3.ª ed.), OTexts.